# Notebook 15 — Feature Engineering for BharatCrop Intelligence

## Objective

Transform the integrated crop and weather dataset into a reliable set of analytical features for agricultural yield prediction, crop recommendation, and risk analysis.

## Scope

- Inspect the integrated dataset and its existing quality-control fields.
- Define the prediction target and distinguish it from input features.
- Engineer crop, season, temporal, and weather-related features where justified by the available data.
- Prevent target leakage, future-information leakage, and inappropriate use of identifiers.
- Handle missing weather coverage without fabricating observations.
- Document feature definitions, assumptions, limitations, and validation checks.
- Save the engineered dataset and a feature-definition reference for downstream notebooks.

## Data Source

Input artifact: `data/processed/weather_regional/apy_weather_integrated_exploratory.csv`

The input combines district-level Area, Production and Yield (APY) records with annual district-level weather data.

## Methodological Constraints

1. **Weather alignment remains unverified.** The existing starting-calendar-year proxy must not be presented as verified crop-season weather.
2. **No fabricated weather data.** Unmatched districts must remain identifiable; missing weather values must not be silently imputed.
3. **Leakage prevention.** Yield, Production, Area, and fields derived from the target must not enter the predictor set unless the intended prediction scenario explicitly permits them.
4. **Temporal integrity.** Any historical or lagged feature must use information available before the prediction period.
5. **Reproducibility.** Feature definitions, transformations, exclusions, and output artifacts must be documented and validated.

## Expected Deliverables

- An engineered dataset with documented feature definitions.
- A feature inventory identifying each feature's purpose, type, and leakage risk.
- Validation checks for missingness, duplicates, data types, and feature consistency.
- Saved outputs for use in subsequent model-training and evaluation notebooks.

## Completion Criteria

The notebook is complete when its feature-engineering decisions are reproducible, leakage risks are explicitly addressed, output artifacts are verified, and unresolved data limitations remain visible.

**Next:** Inspect the integrated dataset before creating any features.

In [1]:
# Notebook 15: Feature Engineering
# Cell 1 — Load the integrated dataset and inspect its structure.
# This cell only reads data; it does not modify the source artifact.

from pathlib import Path
import pandas as pd
import numpy as np

# Define the project root and input artifact.
project_root = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

integrated_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "apy_weather_integrated_exploratory.csv"
)

# Fail clearly if the expected artifact is missing.
if not integrated_path.exists():
    raise FileNotFoundError(
        f"Integrated dataset not found: {integrated_path}"
    )

integrated_df = pd.read_csv(integrated_path, low_memory=False)

print("Dataset shape:", integrated_df.shape)
print("Unique APY years:", integrated_df["Year"].nunique())
print("Unique crops:", integrated_df["Crop"].nunique())
print("Unique seasons:", integrated_df["Season"].nunique())

print("\nColumn inventory:")
for column in integrated_df.columns:
    print(f"- {column}")

print("\nTarget and weather availability:")
print(
    integrated_df[
        ["Yield", "weather_available"]
    ].isna().sum().to_string()
)

print("\nWeather alignment methods:")
print(
    integrated_df["weather_alignment_method"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nFull-season alignment verification:")
print(
    integrated_df["weather_full_season_alignment_verified"]
    .value_counts(dropna=False)
    .to_string()
)

Dataset shape: (376272, 39)
Unique APY years: 12
Unique crops: 19
Unique seasons: 4

Column inventory:
- State
- District
- Crop
- Season
- Crop_Category
- Year
- Area
- Production
- Yield
- QC_AreaZero-ProductionPositive
- QC_AreaPositive-ProductionZero
- QC_NegativeMeasurements
- QC_PositiveYieldWithoutPositiveAreaProduction
- apy_geography_id
- start_year
- weather_proxy_calendar_year
- tmin_mean_c
- tmin_min_c
- tmax_mean_c
- tmax_max_c
- precipitation_total_mm
- precipitation_mean_mm_day
- precipitation_max_mm_day
- tmin_valid_days
- tmax_valid_days
- precipitation_valid_days
- QC_Extreme_Annual_Min_Temperature
- QC_Extreme_Annual_Max_Temperature
- QC_High_Annual_Precipitation
- QC_Extreme_Daily_Precipitation
- QC_Long_District_Grid_Distance
- weather_proxy_available
- _merge
- apy_measurements_complete
- weather_and_apy_complete
- weather_alignment_method
- weather_full_season_alignment_verified
- weather_available
- weather_match_status

Target and weather availability:
Yield   

In [2]:
# Cell 2 — Classify existing columns by analytical role.
# The inventory helps separate predictors, targets, identifiers,
# quality-control fields, and metadata before feature engineering.

column_roles = {
    "target": [
        "Yield",
    ],
    "potential_leakage_or_post-outcome": [
        "Area",
        "Production",
        "apy_measurements_complete",
        "weather_and_apy_complete",
    ],
    "identifiers": [
        "State",
        "District",
        "apy_geography_id",
    ],
    "categorical_predictors": [
        "Crop",
        "Season",
        "Crop_Category",
    ],
    "temporal_fields": [
        "Year",
        "start_year",
        "weather_proxy_calendar_year",
    ],
    "weather_predictors_exploratory": [
        "tmin_mean_c",
        "tmin_min_c",
        "tmax_mean_c",
        "tmax_max_c",
        "precipitation_total_mm",
        "precipitation_mean_mm_day",
        "precipitation_max_mm_day",
        "tmin_valid_days",
        "tmax_valid_days",
        "precipitation_valid_days",
    ],
    "quality_control_fields": [
        "QC_AreaZero-ProductionPositive",
        "QC_AreaPositive-ProductionZero",
        "QC_NegativeMeasurements",
        "QC_PositiveYieldWithoutPositiveAreaProduction",
        "QC_Extreme_Annual_Min_Temperature",
        "QC_Extreme_Annual_Max_Temperature",
        "QC_High_Annual_Precipitation",
        "QC_Extreme_Daily_Precipitation",
        "QC_Long_District_Grid_Distance",
    ],
    "alignment_and_merge_metadata": [
        "_merge",
        "weather_alignment_method",
        "weather_full_season_alignment_verified",
        "weather_proxy_available",
        "weather_available",
        "weather_match_status",
    ],
}

# Check that every declared column exists and that the inventory
# does not accidentally classify any column more than once.
declared_columns = [
    column
    for columns in column_roles.values()
    for column in columns
]

missing_from_dataset = sorted(set(declared_columns) - set(integrated_df.columns))
duplicate_classifications = sorted(
    column
    for column in set(declared_columns)
    if declared_columns.count(column) > 1
)
unclassified_columns = sorted(set(integrated_df.columns) - set(declared_columns))

print("Missing declared columns:", missing_from_dataset)
print("Columns classified more than once:", duplicate_classifications)
print("Unclassified columns:", unclassified_columns)

# Display a readable summary of the planned analytical roles.
role_summary = pd.DataFrame(
    [
        {
            "role": role,
            "column_count": len(columns),
            "columns": ", ".join(columns),
        }
        for role, columns in column_roles.items()
    ]
)

display(role_summary)

assert not missing_from_dataset, "Some declared columns are missing."
assert not duplicate_classifications, "Some columns have multiple roles."
assert not unclassified_columns, "Some dataset columns remain unclassified."

print("\nColumn-role inventory: PASS")

Missing declared columns: []
Columns classified more than once: []
Unclassified columns: []


,role,column_count,columns
0,target,1,Yield
1,potential_leakage_or_post-outcome,4,"Area, Production, apy_measurements_complete, w..."
2,identifiers,3,"State, District, apy_geography_id"
3,categorical_predictors,3,"Crop, Season, Crop_Category"
4,temporal_fields,3,"Year, start_year, weather_proxy_calendar_year"
5,weather_predictors_exploratory,10,"tmin_mean_c, tmin_min_c, tmax_mean_c, tmax_max..."
6,quality_control_fields,9,"QC_AreaZero-ProductionPositive, QC_AreaPositiv..."
7,alignment_and_merge_metadata,6,"_merge, weather_alignment_method, weather_full..."



Column-role inventory: PASS


In [3]:
# Cell 3 — Define the initial yield-prediction contract.
# These rules constrain later feature engineering and help prevent leakage.

prediction_contract = {
    "prediction_task": "district_crop_season_yield_prediction",
    "target_column": "Yield",
    "prediction_unit": "district_crop_season_year",
    "candidate_features": [
        "Crop",
        "Season",
        "Crop_Category",
        "Year",
    ],
    "exploratory_weather_features": [
        "tmin_mean_c",
        "tmin_min_c",
        "tmax_mean_c",
        "tmax_max_c",
        "precipitation_total_mm",
        "precipitation_mean_mm_day",
        "precipitation_max_mm_day",
        "tmin_valid_days",
        "tmax_valid_days",
        "precipitation_valid_days",
    ],
    "excluded_predictors": [
        "Area",
        "Production",
        "Yield",
        "apy_measurements_complete",
        "weather_and_apy_complete",
        "QC_AreaZero-ProductionPositive",
        "QC_AreaPositive-ProductionZero",
        "QC_NegativeMeasurements",
        "QC_PositiveYieldWithoutPositiveAreaProduction",
        "QC_Extreme_Annual_Min_Temperature",
        "QC_Extreme_Annual_Max_Temperature",
        "QC_High_Annual_Precipitation",
        "QC_Extreme_Daily_Precipitation",
        "QC_Long_District_Grid_Distance",
        "_merge",
        "weather_alignment_method",
        "weather_full_season_alignment_verified",
        "weather_proxy_available",
        "weather_available",
        "weather_match_status",
        "apy_geography_id",
        "State",
        "District",
        "start_year",
        "weather_proxy_calendar_year",
    ],
    "weather_status": "EXPLORATORY_ALIGNMENT_UNVERIFIED",
    "evaluation_rule": (
        "Use time-aware evaluation; do not randomly split rows "
        "as the sole evaluation strategy."
    ),
    "causal_claims_allowed": False,
}

# Confirm the target and all proposed input columns exist.
assert prediction_contract["target_column"] in integrated_df.columns

candidate_columns = (
    prediction_contract["candidate_features"]
    + prediction_contract["exploratory_weather_features"]
)

missing_candidates = sorted(
    set(candidate_columns) - set(integrated_df.columns)
)

missing_exclusions = sorted(
    set(prediction_contract["excluded_predictors"])
    - set(integrated_df.columns)
)

print("Prediction task:", prediction_contract["prediction_task"])
print("Target:", prediction_contract["target_column"])
print("Candidate non-weather features:",
      prediction_contract["candidate_features"])
print("Exploratory weather features:",
      len(prediction_contract["exploratory_weather_features"]))
print("Missing candidate columns:", missing_candidates)
print("Missing exclusion columns:", missing_exclusions)
print("Weather status:", prediction_contract["weather_status"])
print("Evaluation rule:", prediction_contract["evaluation_rule"])

assert not missing_candidates
assert not missing_exclusions

print("\nPrediction contract validation: PASS")

Prediction task: district_crop_season_yield_prediction
Target: Yield
Candidate non-weather features: ['Crop', 'Season', 'Crop_Category', 'Year']
Exploratory weather features: 10
Missing candidate columns: []
Missing exclusion columns: []
Weather status: EXPLORATORY_ALIGNMENT_UNVERIFIED
Evaluation rule: Use time-aware evaluation; do not randomly split rows as the sole evaluation strategy.

Prediction contract validation: PASS


In [4]:
# Cell 4 — Audit records for yield modeling.
# Keep the original integrated dataset unchanged.
# This cell measures eligibility; it does not drop or impute records.

# Separate aggregate "Total" records from actual crop-season records.
is_total_record = (
    integrated_df["Season"].astype("string").str.strip().str.casefold()
    .eq("total")
)

# A usable target must be present, finite, and positive.
yield_values = pd.to_numeric(integrated_df["Yield"], errors="coerce")

has_valid_yield = (
    yield_values.notna()
    & np.isfinite(yield_values)
    & yield_values.gt(0)
)

# Identify records with the basic categorical and temporal inputs available.
required_inputs = ["Crop", "Season", "Crop_Category", "Year"]

has_required_inputs = integrated_df[required_inputs].notna().all(axis=1)

# Weather completeness is audited separately because weather alignment
# remains unverified and should not determine basic target eligibility.
weather_columns = prediction_contract["exploratory_weather_features"]
has_complete_weather = (
    integrated_df[weather_columns].notna().all(axis=1)
    & np.isfinite(
        integrated_df[weather_columns]
        .apply(pd.to_numeric, errors="coerce")
    ).all(axis=1)
)

eligibility_summary = pd.DataFrame(
    [
        {
            "check": "All integrated records",
            "records": len(integrated_df),
        },
        {
            "check": "Aggregate Total-season records",
            "records": int(is_total_record.sum()),
        },
        {
            "check": "Non-Total crop-season records",
            "records": int((~is_total_record).sum()),
        },
        {
            "check": "Records with valid positive Yield",
            "records": int(has_valid_yield.sum()),
        },
        {
            "check": "Records with required categorical/time inputs",
            "records": int(has_required_inputs.sum()),
        },
        {
            "check": "Valid Yield and non-Total season",
            "records": int((has_valid_yield & ~is_total_record).sum()),
        },
        {
            "check": "Valid Yield, non-Total season, and required inputs",
            "records": int(
                (has_valid_yield & ~is_total_record & has_required_inputs).sum()
            ),
        },
        {
            "check": "Records with all exploratory weather fields available",
            "records": int(has_complete_weather.sum()),
        },
        {
            "check": "Valid Yield and complete exploratory weather",
            "records": int((has_valid_yield & has_complete_weather).sum()),
        },
    ]
)

display(eligibility_summary)

print("\nYield distribution before eligibility filtering:")
display(
    yield_values.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).to_frame("Yield")
)

print("\nTarget eligibility audit complete; no records were removed.")

,check,records
0,All integrated records,376272
1,Aggregate Total-season records,143028
2,Non-Total crop-season records,233244
3,Records with valid positive Yield,280822
4,Records with required categorical/time inputs,376272
5,Valid Yield and non-Total season,166813
6,"Valid Yield, non-Total season, and required in...",166813
7,Records with all exploratory weather fields av...,320124
8,Valid Yield and complete exploratory weather,241866



Yield distribution before eligibility filtering:


,Yield
count,281239.000000
mean,1815.385992
std,1821.537381
min,0.000000
1%,192.000000
5%,367.000000
25%,829.000000
50%,1436.000000
75%,2504.000000
95%,4214.000000



Target eligibility audit complete; no records were removed.


In [5]:
# Cell 5 — Investigate yield anomalies before defining modeling filters.
# Extreme values are flagged for review, not automatically deleted.

# Convert Yield to numeric for consistent validation.
yield_values = pd.to_numeric(integrated_df["Yield"], errors="coerce")

# Identify invalid targets and extreme positive values.
invalid_yield_mask = (
    yield_values.isna()
    | ~np.isfinite(yield_values)
    | yield_values.le(0)
)

extreme_yield_mask = yield_values.gt(6551)  # Previous audit's 99th percentile.

print("Invalid or non-positive yield records:", int(invalid_yield_mask.sum()))
print("Positive yields above the previous 99th percentile:",
      int(extreme_yield_mask.sum()))

# Inspect the largest values with their crop, season, geography, and
# original area and production measurements.
inspection_columns = [
    "State",
    "District",
    "Crop",
    "Season",
    "Year",
    "Area",
    "Production",
    "Yield",
    "QC_NegativeMeasurements",
    "QC_PositiveYieldWithoutPositiveAreaProduction",
]

largest_yields = (
    integrated_df.loc[
        extreme_yield_mask,
        inspection_columns
    ]
    .sort_values("Yield", ascending=False)
    .head(20)
)

print("\nTop 20 extreme yield records:")
display(largest_yields)

# Summarize target distributions by crop and season.
non_total_valid = integrated_df.loc[
    (~integrated_df["Season"].astype("string").str.strip().str.casefold().eq("total"))
    & yield_values.gt(0)
    & np.isfinite(yield_values)
].copy()

non_total_valid["Yield"] = yield_values.loc[non_total_valid.index]

yield_group_summary = (
    non_total_valid
    .groupby(["Crop", "Season"], dropna=False)["Yield"]
    .agg(
        records="count",
        median="median",
        percentile_99=lambda values: values.quantile(0.99),
        maximum="max",
    )
    .sort_values("maximum", ascending=False)
    .reset_index()
)

print("\nCrop-season groups with the largest observed yields:")
display(yield_group_summary.head(20))

print("\nNo yield records have been removed.")

Invalid or non-positive yield records: 95450
Positive yields above the previous 99th percentile: 2811

Top 20 extreme yield records:


,State,District,Crop,Season,Year,Area,Production,Yield,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction
313625,Andaman And Nicobar Islands,South Andamans,Urad,Total,2023-24,0.00,0.00,385540.0,False,True
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,0.00,0.00,385540.0,False,True
313621,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,2023-24,0.00,0.00,161579.0,False,True
313619,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2023-24,0.00,0.00,161579.0,False,True
313610,Andaman And Nicobar Islands,South Andamans,Moong,Total,2023-24,0.00,0.00,105588.0,False,True
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,0.00,0.00,105588.0,False,True
265536,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,2021-22,0.00,0.01,103571.0,False,True
265537,Madhya Pradesh,Narmadapuram,Other Pulses,Total,2021-22,0.00,0.01,103571.0,False,True
264974,Madhya Pradesh,Harda,Other Pulses,Total,2021-22,0.00,0.01,98889.0,False,True
264973,Madhya Pradesh,Harda,Other Pulses,Rabi,2021-22,0.00,0.01,98889.0,False,True



Crop-season groups with the largest observed yields:


,Crop,Season,records,median,percentile_99,maximum
0,Urad,Rabi,2552,654.0,1919.94,385540.0
1,Total Food Grains,Rabi,8073,2282.0,5371.52,161579.0
2,Moong,Rabi,2244,559.0,1506.57,105588.0
3,Other Pulses,Rabi,6010,946.0,5596.88,103571.0
4,Total Food Grains,Kharif,8231,2006.0,4829.20,44815.0
5,Rice,Kharif,7490,2321.0,4899.21,44815.0
6,Cereals,Kharif,8193,2195.0,5044.44,44815.0
7,Other Pulses,Kharif,2995,738.0,10176.86,27560.0
8,Tur,Kharif,5863,903.0,2277.90,22846.0
9,Jowar,Kharif,3989,1062.0,3108.20,22000.0



No yield records have been removed.


In [6]:
# Cell 6 — Classify yield validity and consistency with Area/Production.
# This is a diagnostic audit only; no rows are removed.

yield_values = pd.to_numeric(integrated_df["Yield"], errors="coerce")
area_values = pd.to_numeric(integrated_df["Area"], errors="coerce")
production_values = pd.to_numeric(integrated_df["Production"], errors="coerce")

# Distinguish missing, non-finite, zero, negative, and positive targets.
target_status = pd.Series("POSITIVE_FINITE", index=integrated_df.index, dtype="string")
target_status.loc[yield_values.isna()] = "MISSING_OR_NONNUMERIC"
target_status.loc[yield_values.notna() & ~np.isfinite(yield_values)] = "NONFINITE"
target_status.loc[yield_values.eq(0)] = "ZERO"
target_status.loc[yield_values.lt(0)] = "NEGATIVE"

print("Yield target status:")
display(
    target_status.value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="records")
)

# Check yield records that cannot be reconciled with positive Area and Production.
area_zero_production_positive = area_values.eq(0) & production_values.gt(0)
area_positive_production_zero = area_values.gt(0) & production_values.eq(0)
positive_yield_without_valid_inputs = (
    yield_values.gt(0)
    & np.isfinite(yield_values)
    & (
        area_values.isna()
        | production_values.isna()
        | area_values.le(0)
        | production_values.le(0)
    )
)

print("\nMeasurement consistency checks:")
print("Area = 0 and Production > 0:", int(area_zero_production_positive.sum()))
print("Area > 0 and Production = 0:", int(area_positive_production_zero.sum()))
print(
    "Positive finite Yield without positive Area and Production:",
    int(positive_yield_without_valid_inputs.sum())
)

# Compare the newly calculated issue with the existing QC flag.
existing_flag = (
    integrated_df["QC_PositiveYieldWithoutPositiveAreaProduction"]
    .fillna(False)
    .astype(bool)
)

print("\nExisting QC flag comparison:")
print("Flagged records:", int(existing_flag.sum()))
print(
    "Newly detected but not flagged:",
    int((positive_yield_without_valid_inputs & ~existing_flag).sum())
)
print(
    "Flagged but not detected by this check:",
    int((existing_flag & ~positive_yield_without_valid_inputs).sum())
)

# Show the largest extreme yields with the original measurements and QC flags.
extreme_mask = yield_values.gt(6551) & np.isfinite(yield_values)

extreme_audit = integrated_df.loc[
    extreme_mask,
    [
        "State", "District", "Crop", "Season", "Year",
        "Area", "Production", "Yield",
        "QC_AreaZero-ProductionPositive",
        "QC_AreaPositive-ProductionZero",
        "QC_PositiveYieldWithoutPositiveAreaProduction",
    ],
].sort_values("Yield", ascending=False).head(15)

print("\nLargest extreme-yield records:")
display(extreme_audit)

print("\nAudit complete: source dataset unchanged.")

Yield target status:


,status,records
0,POSITIVE_FINITE,280822
1,MISSING_OR_NONNUMERIC,95033
2,ZERO,417



Measurement consistency checks:
Area = 0 and Production > 0: 9358
Area > 0 and Production = 0: 6582
Positive finite Yield without positive Area and Production: 94235

Existing QC flag comparison:
Flagged records: 94235
Newly detected but not flagged: 0
Flagged but not detected by this check: 0

Largest extreme-yield records:


,State,District,Crop,Season,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_PositiveYieldWithoutPositiveAreaProduction
313625,Andaman And Nicobar Islands,South Andamans,Urad,Total,2023-24,0.00,0.00,385540.0,False,False,True
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,0.00,0.00,385540.0,False,False,True
313621,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,2023-24,0.00,0.00,161579.0,False,False,True
313619,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2023-24,0.00,0.00,161579.0,False,False,True
313610,Andaman And Nicobar Islands,South Andamans,Moong,Total,2023-24,0.00,0.00,105588.0,False,False,True
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,0.00,0.00,105588.0,False,False,True
265536,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,2021-22,0.00,0.01,103571.0,True,False,True
265537,Madhya Pradesh,Narmadapuram,Other Pulses,Total,2021-22,0.00,0.01,103571.0,True,False,True
264974,Madhya Pradesh,Harda,Other Pulses,Total,2021-22,0.00,0.01,98889.0,True,False,True
264973,Madhya Pradesh,Harda,Other Pulses,Rabi,2021-22,0.00,0.01,98889.0,True,False,True



Audit complete: source dataset unchanged.


In [7]:
# Cell 7 — Create provisional eligibility flags for yield modeling.
# These flags are transparent and reversible; no rows are deleted.

feature_df = integrated_df.copy()

# Normalize the season label for reliable identification of aggregate records.
season_normalized = (
    feature_df["Season"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Convert measurement columns to numeric for consistent checks.
yield_numeric = pd.to_numeric(feature_df["Yield"], errors="coerce")
area_numeric = pd.to_numeric(feature_df["Area"], errors="coerce")
production_numeric = pd.to_numeric(feature_df["Production"], errors="coerce")

# Flag records with a finite, positive yield target.
feature_df["FE_ValidPositiveYield"] = (
    yield_numeric.notna()
    & np.isfinite(yield_numeric)
    & yield_numeric.gt(0)
)

# Aggregate totals are not individual growing seasons.
feature_df["FE_IsAggregateTotal"] = season_normalized.eq("total")

# Flag yield records whose area or production inputs are missing or non-positive.
feature_df["FE_InvalidAreaProductionForYield"] = (
    area_numeric.isna()
    | production_numeric.isna()
    | area_numeric.le(0)
    | production_numeric.le(0)
)

# Flag yields above the global 99th percentile found in the earlier audit.
# This is a review flag, NOT an automatic exclusion rule.
feature_df["FE_ExtremeYieldReview"] = (
    feature_df["FE_ValidPositiveYield"]
    & yield_numeric.gt(6551)
)

# Basic target eligibility excludes invalid targets and aggregate totals.
feature_df["FE_BasicTargetEligible"] = (
    feature_df["FE_ValidPositiveYield"]
    & ~feature_df["FE_IsAggregateTotal"]
)

# Stricter eligibility additionally requires positive Area and Production.
# This is provisional because source units and yield derivation still need verification.
feature_df["FE_ConsistentMeasurementEligible"] = (
    feature_df["FE_BasicTargetEligible"]
    & ~feature_df["FE_InvalidAreaProductionForYield"]
)

eligibility_counts = pd.DataFrame([
    {
        "rule": "All records",
        "records": len(feature_df),
    },
    {
        "rule": "Valid positive finite Yield",
        "records": int(feature_df["FE_ValidPositiveYield"].sum()),
    },
    {
        "rule": "Valid positive Yield, non-Total season",
        "records": int(feature_df["FE_BasicTargetEligible"].sum()),
    },
    {
        "rule": "Basic eligible + positive Area and Production",
        "records": int(feature_df["FE_ConsistentMeasurementEligible"].sum()),
    },
    {
        "rule": "Basic eligible + extreme-yield review flag",
        "records": int(
            (
                feature_df["FE_BasicTargetEligible"]
                & feature_df["FE_ExtremeYieldReview"]
            ).sum()
        ),
    },
])

display(eligibility_counts)

print("\nEligibility flags created:", [
    "FE_ValidPositiveYield",
    "FE_IsAggregateTotal",
    "FE_InvalidAreaProductionForYield",
    "FE_ExtremeYieldReview",
    "FE_BasicTargetEligible",
    "FE_ConsistentMeasurementEligible",
])

print("\nOriginal integrated dataset unchanged:", len(integrated_df) == 376272)

,rule,records
0,All records,376272
1,Valid positive finite Yield,280822
2,"Valid positive Yield, non-Total season",166813
3,Basic eligible + positive Area and Production,107267
4,Basic eligible + extreme-yield review flag,1831



Eligibility flags created: ['FE_ValidPositiveYield', 'FE_IsAggregateTotal', 'FE_InvalidAreaProductionForYield', 'FE_ExtremeYieldReview', 'FE_BasicTargetEligible', 'FE_ConsistentMeasurementEligible']

Original integrated dataset unchanged: True


In [8]:
# Cell 8 — Compare basic and stricter eligibility by crop and season.
# This is a coverage audit; it does not remove observations.

coverage_by_group = (
    feature_df.loc[~feature_df["FE_IsAggregateTotal"]]
    .groupby(["Crop", "Season"], dropna=False)
    .agg(
        total_records=("Yield", "size"),
        basic_eligible=("FE_BasicTargetEligible", "sum"),
        measurement_eligible=("FE_ConsistentMeasurementEligible", "sum"),
        extreme_yield_review=("FE_ExtremeYieldReview", "sum"),
    )
    .reset_index()
)

# Calculate retention rates relative to all records in each crop-season group.
coverage_by_group["basic_eligibility_pct"] = (
    100 * coverage_by_group["basic_eligible"]
    / coverage_by_group["total_records"]
)

coverage_by_group["measurement_eligibility_pct"] = (
    100 * coverage_by_group["measurement_eligible"]
    / coverage_by_group["total_records"]
)

coverage_by_group["records_lost_to_measurement_rule"] = (
    coverage_by_group["basic_eligible"]
    - coverage_by_group["measurement_eligible"]
)

print("Crop-season groups with the lowest retention under the stricter rule:")
display(
    coverage_by_group.sort_values(
        ["measurement_eligibility_pct", "total_records"],
        ascending=[True, False],
    ).head(20).round(2)
)

print("\nOverall retention among basic eligible records:")
basic_count = int(feature_df["FE_BasicTargetEligible"].sum())
strict_count = int(feature_df["FE_ConsistentMeasurementEligible"].sum())

print("Basic eligible:", basic_count)
print("Measurement eligible:", strict_count)
print(
    "Retained under stricter rule (%):",
    round(100 * strict_count / basic_count, 2) if basic_count else np.nan,
)

print("\nTotal basic-eligible records excluded by the stricter rule:",
      basic_count - strict_count)

print("\nNo rows were removed from feature_df.")

Crop-season groups with the lowest retention under the stricter rule:


,Crop,Season,total_records,basic_eligible,measurement_eligible,extreme_yield_review,basic_eligibility_pct,measurement_eligibility_pct,records_lost_to_measurement_rule
9,Gram,Kharif,324,32,0,0,9.88,0.0,32
51,Wheat,Summer,276,94,0,0,34.06,0.0,94
49,Wheat,Kharif,168,17,0,0,10.12,0.0,17
5,Barley,Summer,108,0,0,0,0.0,0.0,0
16,Lentil,Summer,48,0,0,0,0.0,0.0,0
28,Other Pulses,Summer,504,257,3,0,50.99,0.6,254
40,Small Millets,Summer,408,60,3,0,14.71,0.74,57
45,Tur,Rabi,1560,604,62,0,38.72,3.97,542
39,Small Millets,Rabi,1824,539,97,0,29.55,5.32,442
14,Lentil,Kharif,600,68,36,0,11.33,6.0,32



Overall retention among basic eligible records:
Basic eligible: 166813
Measurement eligible: 107267
Retained under stricter rule (%): 64.3

Total basic-eligible records excluded by the stricter rule: 59546

No rows were removed from feature_df.


In [9]:
# Cell 9 — Diagnose the relationship between reported yield,
# area, and production without assuming undocumented units.
# No observations are removed.

area = pd.to_numeric(feature_df["Area"], errors="coerce")
production = pd.to_numeric(feature_df["Production"], errors="coerce")
yield_reported = pd.to_numeric(feature_df["Yield"], errors="coerce")

# Restrict arithmetic comparison to records where all three values
# are finite and strictly positive.
arithmetic_check_mask = (
    np.isfinite(area)
    & np.isfinite(production)
    & np.isfinite(yield_reported)
    & area.gt(0)
    & production.gt(0)
    & yield_reported.gt(0)
)

arithmetic_check = feature_df.loc[
    arithmetic_check_mask,
    ["Crop", "Season", "Year", "Area", "Production", "Yield"]
].copy()

# Compute the raw ratio. Do not call it a valid yield until units
# and conversion factors have been confirmed from source documentation.
arithmetic_check["production_per_area_raw"] = (
    production.loc[arithmetic_check.index]
    / area.loc[arithmetic_check.index]
)

arithmetic_check["yield_to_raw_ratio"] = (
    yield_reported.loc[arithmetic_check.index]
    / arithmetic_check["production_per_area_raw"]
)

print("Records with positive finite Area, Production, and Yield:",
      len(arithmetic_check))

print("\nReported Yield / (Production / Area) ratio distribution:")
display(
    arithmetic_check["yield_to_raw_ratio"]
    .replace([np.inf, -np.inf], np.nan)
    .describe(percentiles=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99])
    .to_frame("ratio")
)

print("\nMost common ratio magnitudes (rounded to 3 significant digits):")
ratio_magnitude = (
    np.log10(arithmetic_check["yield_to_raw_ratio"].abs())
    .round(1)
    .value_counts()
    .head(15)
    .rename_axis("log10_ratio")
    .reset_index(name="records")
)
display(ratio_magnitude)

print("\nLargest ratios for manual inspection:")
display(
    arithmetic_check.sort_values(
        "yield_to_raw_ratio", ascending=False
    ).head(10)
)

print("\nSmallest ratios for manual inspection:")
display(
    arithmetic_check.sort_values(
        "yield_to_raw_ratio", ascending=True
    ).head(10)
)

print("\nDiagnostic complete; no rows were removed.")

Records with positive finite Area, Production, and Yield: 186587

Reported Yield / (Production / Area) ratio distribution:


,ratio
count,186587.000000
mean,1016.889596
std,178.238372
min,342.000000
1%,599.000000
10%,893.333333
25%,985.714286
50%,1000.000000
75%,1015.444444
90%,1132.000000



Most common ratio magnitudes (rounded to 3 significant digits):


,log10_ratio,records
0,3.0,148526
1,3.1,13418
2,2.9,13085
3,3.2,4408
4,2.8,4121
5,3.3,1412
6,2.7,922
7,3.4,426
8,2.6,248
9,3.5,12



Largest ratios for manual inspection:


,Crop,Season,Year,Area,Production,Yield,production_per_area_raw,yield_to_raw_ratio
281886,Maize,Total,2021-22,0.01,0.01,2935.0,1.0,2935.0
281894,Nutri/Coarse Cereals,Total,2021-22,0.01,0.01,2935.0,1.0,2935.0
316723,Maize,Kharif,2023-24,0.01,0.01,2911.0,1.0,2911.0
167828,Nutri/Coarse Cereals,Rabi,2018-19,0.01,0.01,2909.0,1.0,2909.0
167808,Cereals,Rabi,2018-19,0.01,0.01,2905.0,1.0,2905.0
247199,Maize,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0
247189,Cereals,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0
247206,Nutri/Coarse Cereals,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0
316276,Rice,Summer,2023-24,0.01,0.01,2883.0,1.0,2883.0
316255,Cereals,Summer,2023-24,0.01,0.01,2883.0,1.0,2883.0



Smallest ratios for manual inspection:


,Crop,Season,Year,Area,Production,Yield,production_per_area_raw,yield_to_raw_ratio
239291,Other Pulses,Rabi,2020-21,0.01,0.01,342.0,1.0,342.0
239292,Other Pulses,Total,2020-21,0.01,0.01,342.0,1.0,342.0
288222,Urad,Kharif,2022-23,0.01,0.01,347.0,1.0,347.0
288223,Urad,Total,2022-23,0.01,0.01,347.0,1.0,347.0
37597,Lentil,Total,2014-15,0.01,0.01,352.0,1.0,352.0
37596,Lentil,Rabi,2014-15,0.01,0.01,352.0,1.0,352.0
170340,Urad,Kharif,2018-19,0.01,0.01,353.0,1.0,353.0
170342,Urad,Total,2018-19,0.01,0.01,353.0,1.0,353.0
324890,Jowar,Kharif,2023-24,0.01,0.01,353.0,1.0,353.0
103178,Urad,Kharif,2016-17,0.01,0.01,355.0,1.0,355.0



Diagnostic complete; no rows were removed.


In [10]:
# Cell 10 — Quantify the possible yield-unit conversion pattern.
# A ratio near 1,000 is evidence of a scale pattern, not proof of units.

ratio = arithmetic_check["yield_to_raw_ratio"]

# Measure proximity to 1,000 using relative deviation.
arithmetic_check["ratio_relative_deviation_from_1000"] = (
    (ratio - 1000).abs() / 1000
)

ratio_tolerance_summary = pd.DataFrame([
    {
        "criterion": "Within 1% of 1,000",
        "records": int((arithmetic_check["ratio_relative_deviation_from_1000"] <= 0.01).sum()),
    },
    {
        "criterion": "Within 5% of 1,000",
        "records": int((arithmetic_check["ratio_relative_deviation_from_1000"] <= 0.05).sum()),
    },
    {
        "criterion": "Within 10% of 1,000",
        "records": int((arithmetic_check["ratio_relative_deviation_from_1000"] <= 0.10).sum()),
    },
    {
        "criterion": "More than 25% from 1,000",
        "records": int((arithmetic_check["ratio_relative_deviation_from_1000"] > 0.25).sum()),
    },
])

ratio_tolerance_summary["percent_of_comparison_records"] = (
    100 * ratio_tolerance_summary["records"] / len(arithmetic_check)
)

display(ratio_tolerance_summary.round(2))

# Check whether the median ratio pattern is stable across reporting years.
ratio_by_year = (
    arithmetic_check
    .groupby("Year")["yield_to_raw_ratio"]
    .agg(
        records="count",
        median_ratio="median",
        percentile_10=lambda values: values.quantile(0.10),
        percentile_90=lambda values: values.quantile(0.90),
    )
    .reset_index()
)

print("\nRatio by APY reporting year:")
display(ratio_by_year.round(2))

# Check whether the pattern differs across crops with sufficient observations.
ratio_by_crop = (
    arithmetic_check
    .groupby("Crop")["yield_to_raw_ratio"]
    .agg(
        records="count",
        median_ratio="median",
        percentile_10=lambda values: values.quantile(0.10),
        percentile_90=lambda values: values.quantile(0.90),
    )
    .query("records >= 100")
    .sort_values("median_ratio")
    .reset_index()
)

print("\nCrop-level ratio summary (at least 100 records):")
display(ratio_by_crop.round(2))

print("\nNo unit conversion has been applied; the source data remains unchanged.")

,criterion,records,percent_of_comparison_records
0,"Within 1% of 1,000",82798,44.38
1,"Within 5% of 1,000",125885,67.47
2,"Within 10% of 1,000",144879,77.65
3,"More than 25% from 1,000",18209,9.76



Ratio by APY reporting year:


,Year,records,median_ratio,percentile_10,percentile_90
0,2013-14,14340,1000.00,884.30,1127.50
1,2014-15,14981,1000.00,900.00,1108.80
2,2015-16,14859,1000.00,894.00,1117.00
3,2016-17,15416,1000.00,902.00,1125.27
4,2017-18,15998,1000.00,894.00,1132.00
5,2018-19,15468,1000.00,888.44,1138.00
6,2019-20,15757,1000.00,886.29,1135.62
7,2020-21,15751,1000.00,890.00,1134.38
8,2021-22,15544,1000.00,887.52,1150.20
9,2022-23,16088,1000.27,899.00,1149.77



Crop-level ratio summary (at least 100 records):


,Crop,records,median_ratio,percentile_10,percentile_90
0,Urad,7904,987.00,677.30,1196.00
1,Moong,6473,993.00,694.20,1200.00
2,Other Pulses,6971,997.50,749.50,1298.25
3,Lentil,4128,999.00,743.00,1197.00
4,Ragi,2038,1000.00,800.00,1318.00
5,Shree Anna /Nutri Cereals,9876,1000.00,867.62,1154.00
6,Jowar,5169,1000.00,858.33,1182.40
7,Gram,6852,1000.00,898.00,1122.94
8,Tur,6286,1000.00,798.00,1224.00
9,Wheat,9938,1000.00,980.00,1026.00



No unit conversion has been applied; the source data remains unchanged.


In [11]:
# Cell 11 — Document measurement-scale findings and flag records for review.
# This is a diagnostic artifact, not a correction or exclusion step.

measurement_review = arithmetic_check[
    [
        "Crop",
        "Season",
        "Year",
        "Area",
        "Production",
        "Yield",
        "production_per_area_raw",
        "yield_to_raw_ratio",
        "ratio_relative_deviation_from_1000",
    ]
].copy()

measurement_review["FE_RatioReviewFlag"] = (
    measurement_review["ratio_relative_deviation_from_1000"] > 0.10
)

measurement_review["FE_RatioReviewReason"] = np.where(
    measurement_review["FE_RatioReviewFlag"],
    "Ratio differs by more than 10% from provisional 1000 scale",
    "Within 10% of provisional 1000 scale",
)

measurement_review_summary = pd.DataFrame(
    [
        {
            "metric": "Records evaluated",
            "value": len(measurement_review),
        },
        {
            "metric": "Records within 10% of ratio 1000",
            "value": int((~measurement_review["FE_RatioReviewFlag"]).sum()),
        },
        {
            "metric": "Records flagged for ratio review",
            "value": int(measurement_review["FE_RatioReviewFlag"].sum()),
        },
        {
            "metric": "Source units officially verified",
            "value": "No",
        },
        {
            "metric": "Measurements corrected or removed",
            "value": "No",
        },
    ]
)

display(measurement_review_summary)

print("\nExamples flagged for manual review:")
display(
    measurement_review.loc[measurement_review["FE_RatioReviewFlag"]]
    .sort_values("ratio_relative_deviation_from_1000", ascending=False)
    .head(15)
    .round(3)
)

# Preserve the audit artifact without modifying feature_df or integrated_df.
review_output_path = (
    project_root
    / "data"
    / "processed"
    / "weather_regional"
    / "yield_area_production_ratio_review.csv"
)

review_output_path.parent.mkdir(parents=True, exist_ok=True)
measurement_review.to_csv(review_output_path, index=False)

print(f"\nSaved measurement review: {review_output_path}")
print("No target values were transformed and no observations were removed.")

,metric,value
0,Records evaluated,186587
1,Records within 10% of ratio 1000,144879
2,Records flagged for ratio review,41708
3,Source units officially verified,No
4,Measurements corrected or removed,No



Examples flagged for manual review:


,Crop,Season,Year,Area,Production,Yield,production_per_area_raw,yield_to_raw_ratio,ratio_relative_deviation_from_1000,FE_RatioReviewFlag,FE_RatioReviewReason
281886,Maize,Total,2021-22,0.01,0.01,2935.0,1.0,2935.0,1.935,True,Ratio differs by more than 10% from provisiona...
281894,Nutri/Coarse Cereals,Total,2021-22,0.01,0.01,2935.0,1.0,2935.0,1.935,True,Ratio differs by more than 10% from provisiona...
316723,Maize,Kharif,2023-24,0.01,0.01,2911.0,1.0,2911.0,1.911,True,Ratio differs by more than 10% from provisiona...
167828,Nutri/Coarse Cereals,Rabi,2018-19,0.01,0.01,2909.0,1.0,2909.0,1.909,True,Ratio differs by more than 10% from provisiona...
167808,Cereals,Rabi,2018-19,0.01,0.01,2905.0,1.0,2905.0,1.905,True,Ratio differs by more than 10% from provisiona...
247189,Cereals,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0,1.892,True,Ratio differs by more than 10% from provisiona...
247206,Nutri/Coarse Cereals,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0,1.892,True,Ratio differs by more than 10% from provisiona...
247199,Maize,Summer,2020-21,0.01,0.01,2892.0,1.0,2892.0,1.892,True,Ratio differs by more than 10% from provisiona...
316255,Cereals,Summer,2023-24,0.01,0.01,2883.0,1.0,2883.0,1.883,True,Ratio differs by more than 10% from provisiona...
316276,Rice,Summer,2023-24,0.01,0.01,2883.0,1.0,2883.0,1.883,True,Ratio differs by more than 10% from provisiona...



Saved measurement review: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\weather_regional\yield_area_production_ratio_review.csv
No target values were transformed and no observations were removed.


In [12]:
# Cell 12 — Locate original APY source files and dataset documentation.
# Search likely source/documentation filenames within the project directory.

from pathlib import Path

search_roots = [
    project_root / "data",
    project_root / "docs",
]

keywords = (
    "apy",
    "agricultural",
    "yield",
    "production",
    "crop",
    "dictionary",
    "metadata",
    "source",
    "readme",
)

supported_extensions = {
    ".csv", ".xlsx", ".xls", ".json", ".parquet",
    ".txt", ".md", ".pdf", ".html",
}

source_candidates = []

for root in search_roots:
    if not root.exists():
        print(f"Search directory not found: {root}")
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in supported_extensions:
            continue

        name = path.name.lower()
        if any(keyword in name for keyword in keywords):
            source_candidates.append(path)

source_candidates = sorted(set(source_candidates), key=lambda p: str(p).lower())

print(f"Potential source/documentation files found: {len(source_candidates)}")

for path in source_candidates[:100]:
    print(f"{path.relative_to(project_root)}  [{path.suffix.lower()}]")

if len(source_candidates) > 100:
    print(f"... and {len(source_candidates) - 100} additional files.")

Search directory not found: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\docs
Potential source/documentation files found: 20
data\processed\.ipynb_checkpoints\historical_food_grains_apy_2013-14_to_2024-25-checkpoint.csv  [.csv]
data\processed\historical_food_grains_apy_2013-14_to_2024-25.csv  [.csv]
data\processed\historical_food_grains_apy_cleaned_flagged.csv  [.csv]
data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv  [.csv]
data\processed\weather_regional\apy_districts_without_weather_coverage.csv  [.csv]
data\processed\weather_regional\apy_weather_integrated_exploratory.csv  [.csv]
data\processed\weather_regional\apy_weather_integration_summary.csv  [.csv]
data\processed\weather_regional\apy_weather_proxy_district_year_coverage.csv  [.csv]
data\processed\weather_regional\crop_weather_integration_validation_summary.csv  [.csv]
data\processed\weather_regional\yield_area_production_ratio_review.csv  [.csv]
data\raw\weather_pilot_2013_2024\APY-GEO-0005_weat

In [13]:
# Cell 13 — Inspect APY source variants for measurement definitions.
# Identify whether the original or transformed files preserve unit metadata.

apy_files = [
    project_root / "data" / "processed" /
    "historical_food_grains_apy_2013-14_to_2024-25.csv",

    project_root / "data" / "processed" /
    "historical_food_grains_apy_cleaned_flagged.csv",

    project_root / "data" / "processed" /
    "historical_food_grains_apy_long_2013-14_to_2024-25.csv",
]

unit_keywords = ("unit", "yield", "production", "area", "source", "note", "definition")

for path in apy_files:
    print("\n" + "=" * 90)
    print(f"FILE: {path.name}")

    if not path.exists():
        print("File not found.")
        continue

    # Read a small sample first; no source file is changed.
    sample_df = pd.read_csv(path, nrows=5, low_memory=False)

    print(f"Sample shape: {sample_df.shape}")
    print("Columns and data types:")

    for column, dtype in sample_df.dtypes.items():
        print(f"  {column}: {dtype}")

    relevant_columns = [
        column for column in sample_df.columns
        if any(keyword in column.lower() for keyword in unit_keywords)
    ]

    print("\nUnit/measurement-related columns:")
    print(relevant_columns if relevant_columns else "No explicitly named unit/definition columns found.")

    print("\nSample records:")
    display(sample_df.head(3))


FILE: historical_food_grains_apy_2013-14_to_2024-25.csv
Sample shape: (5, 40)
Columns and data types:
  State: object
  District: object
  Crop: object
  Season: object
  Area-2013-14: float64
  Production-2013-14: float64
  Yield-2013-14: float64
  Area-2014-15: float64
  Production-2014-15: float64
  Yield-2014-15: float64
  Area-2015-16: float64
  Production-2015-16: float64
  Yield-2015-16: float64
  Area-2016-17: float64
  Production-2016-17: float64
  Yield-2016-17: float64
  Area-2017-18: float64
  Production-2017-18: float64
  Yield-2017-18: float64
  Area-2018-19: float64
  Production-2018-19: float64
  Yield-2018-19: float64
  Area-2019-20: float64
  Production-2019-20: float64
  Yield-2019-20: float64
  Area-2020-21: float64
  Production-2020-21: float64
  Yield-2020-21: float64
  Area-2021-22: float64
  Production-2021-22: float64
  Yield-2021-22: float64
  Area-2022-23: float64
  Production-2022-23: float64
  Yield-2022-23: float64
  Area-2023-24: float64
  Production-202

,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,...,Yield-2021-22,Area-2022-23,Production-2022-23,Yield-2022-23,Area-2023-24,Production-2023-24,Yield-2023-24,Area-2024-25,Production-2024-25,Yield-2024-25
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,...,NaN,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,...,1133.0,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0



FILE: historical_food_grains_apy_cleaned_flagged.csv
Sample shape: (5, 89)
Columns and data types:
  State: object
  District: object
  Crop: object
  Season: object
  Area-2013-14: float64
  Production-2013-14: float64
  Yield-2013-14: float64
  Area-2014-15: float64
  Production-2014-15: float64
  Yield-2014-15: float64
  Area-2015-16: float64
  Production-2015-16: float64
  Yield-2015-16: float64
  Area-2016-17: float64
  Production-2016-17: float64
  Yield-2016-17: float64
  Area-2017-18: float64
  Production-2017-18: float64
  Yield-2017-18: float64
  Area-2018-19: float64
  Production-2018-19: float64
  Yield-2018-19: float64
  Area-2019-20: float64
  Production-2019-20: float64
  Yield-2019-20: float64
  Area-2020-21: float64
  Production-2020-21: float64
  Yield-2020-21: float64
  Area-2021-22: float64
  Production-2021-22: float64
  Yield-2021-22: float64
  Area-2022-23: float64
  Production-2022-23: float64
  Yield-2022-23: float64
  Area-2023-24: float64
  Production-2023-2

,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,...,QC-2022-23-NegativeMeasurements,QC-2022-23-PositiveYieldWithoutPositiveAreaProduction,QC-2023-24-AreaZero-ProductionPositive,QC-2023-24-AreaPositive-ProductionZero,QC-2023-24-NegativeMeasurements,QC-2023-24-PositiveYieldWithoutPositiveAreaProduction,QC-2024-25-AreaZero-ProductionPositive,QC-2024-25-AreaPositive-ProductionZero,QC-2024-25-NegativeMeasurements,QC-2024-25-PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,...,False,True,False,False,False,True,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,False,False,False,False,False,False,False,False,False,False
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,...,False,True,False,False,False,True,False,False,False,True



FILE: historical_food_grains_apy_long_2013-14_to_2024-25.csv
Sample shape: (5, 13)
Columns and data types:
  State: object
  District: object
  Crop: object
  Season: object
  Crop_Category: object
  Year: object
  Area: float64
  Production: float64
  Yield: float64
  QC_AreaZero-ProductionPositive: bool
  QC_AreaPositive-ProductionZero: bool
  QC_NegativeMeasurements: bool
  QC_PositiveYieldWithoutPositiveAreaProduction: bool

Unit/measurement-related columns:
['Area', 'Production', 'Yield', 'QC_AreaZero-ProductionPositive', 'QC_AreaPositive-ProductionZero', 'QC_PositiveYieldWithoutPositiveAreaProduction']

Sample records:


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,False,True
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,False,True


In [14]:
# Cell 14 — Inspect source metadata and representative APY measurements.
# This helps identify provenance clues and compare the long-format data
# against the integrated dataset without assuming the units.

apy_long_path = (
    project_root / "data" / "processed" /
    "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
)

apy_long_df = pd.read_csv(apy_long_path, low_memory=False)

print("Long-format APY dataset")
print(f"Shape: {apy_long_df.shape}")
print(f"Years: {apy_long_df['Year'].nunique()}")
print(f"Crops: {apy_long_df['Crop'].nunique()}")
print(f"Seasons: {sorted(apy_long_df['Season'].dropna().unique().tolist())}")

print("\nMeasurement summary:")
display(
    apy_long_df[["Area", "Production", "Yield"]]
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .round(3)
)

print("\nRecords where Area and Production are both zero but Yield is positive:")
display(
    apy_long_df.loc[
        (apy_long_df["Area"] == 0)
        & (apy_long_df["Production"] == 0)
        & (apy_long_df["Yield"] > 0),
        ["State", "District", "Crop", "Season", "Year",
         "Area", "Production", "Yield"]
    ].head(10)
)

print("\nDataset file metadata:")
for path in [
    project_root / "data" / "processed" /
    "historical_food_grains_apy_2013-14_to_2024-25.csv",
    apy_long_path,
]:
    stat = path.stat()
    print(f"{path.name}: {stat.st_size:,} bytes")

print("\nNo records were changed.")

Long-format APY dataset
Shape: (376272, 13)
Years: 12
Crops: 19
Seasons: ['Kharif', 'Rabi', 'Summer', 'Total']

Measurement summary:


,Area,Production,Yield
count,282089.000,281242.000,281239.000
mean,0.354,0.859,1815.386
std,0.788,2.104,1821.537
min,0.000,0.000,0.000
1%,0.000,0.000,192.000
5%,0.000,0.000,367.000
50%,0.030,0.040,1436.000
95%,1.880,5.000,4214.000
99%,3.690,10.540,6551.000
max,12.680,31.880,385540.000



Records where Area and Production are both zero but Yield is positive:


,State,District,Crop,Season,Year,Area,Production,Yield
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,2013-14,0.0,0.0,2377.0
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,2013-14,0.0,0.0,526.0
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,2013-14,0.0,0.0,923.0
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,2013-14,0.0,0.0,526.0
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,2013-14,0.0,0.0,526.0
7,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Rabi,2013-14,0.0,0.0,526.0
8,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Total,2013-14,0.0,0.0,526.0
9,Andaman And Nicobar Islands,Nicobars,Other Pulses,Rabi,2013-14,0.0,0.0,3865.0
10,Andaman And Nicobar Islands,Nicobars,Other Pulses,Total,2013-14,0.0,0.0,3865.0
11,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,2013-14,0.0,0.0,2377.0



Dataset file metadata:
historical_food_grains_apy_2013-14_to_2024-25.csv: 5,982,106 bytes
historical_food_grains_apy_long_2013-14_to_2024-25.csv: 43,549,559 bytes

No records were changed.


In [15]:
# Cell 15 — Select real APY records for official-source verification.
# Choose examples that actually exist in our local dataset.

apy_long_df["Year"] = apy_long_df["Year"].astype(str)

# Case A: Records with positive Area, Production, and Yield.
valid_measurements = apy_long_df.loc[
    (apy_long_df["Area"] > 0)
    & (apy_long_df["Production"] > 0)
    & (apy_long_df["Yield"] > 0)
].copy()

# Case B: Records with tiny positive Area and Production.
tiny_measurements = apy_long_df.loc[
    (apy_long_df["Area"] == 0.01)
    & (apy_long_df["Production"] == 0.01)
    & (apy_long_df["Yield"] > 0)
].copy()

# Prefer recent years for easier comparison in the official portal.
year_priority = ["2024-25", "2023-24", "2022-23", "2021-22"]

def show_candidates(df, label, count=10):
    print(f"\n{label}: {len(df):,} matching records")

    if df.empty:
        print("No candidates found.")
        return

    df["year_priority"] = df["Year"].map(
        {year: index for index, year in enumerate(year_priority)}
    ).fillna(len(year_priority))

    display(
        df.sort_values(["year_priority", "State", "District", "Crop"])
        .drop(columns="year_priority")
        .head(count)[
            ["State", "District", "Crop", "Season", "Year",
             "Area", "Production", "Yield"]
        ]
    )

show_candidates(valid_measurements, "Positive Area, Production, and Yield")
show_candidates(tiny_measurements, "Area = 0.01 and Production = 0.01")


Positive Area, Production, and Yield: 186,587 matching records


,State,District,Crop,Season,Year,Area,Production,Yield
344935,Andaman And Nicobar Islands,North And Middle Andaman,Cereals,Kharif,2024-25,0.04,0.09,2062.0
344937,Andaman And Nicobar Islands,North And Middle Andaman,Cereals,Total,2024-25,0.04,0.09,2064.0
344948,Andaman And Nicobar Islands,North And Middle Andaman,Rice,Kharif,2024-25,0.04,0.09,2062.0
344949,Andaman And Nicobar Islands,North And Middle Andaman,Rice,Total,2024-25,0.04,0.09,2062.0
344950,Andaman And Nicobar Islands,North And Middle Andaman,Total Food Grains,Kharif,2024-25,0.04,0.09,2062.0
344952,Andaman And Nicobar Islands,North And Middle Andaman,Total Food Grains,Total,2024-25,0.05,0.09,1805.0
344984,Andhra Pradesh,Alluri Sitharama Raju,Cereals,Kharif,2024-25,0.74,2.00,2691.0
344985,Andhra Pradesh,Alluri Sitharama Raju,Cereals,Rabi,2024-25,0.02,0.07,4003.0
344986,Andhra Pradesh,Alluri Sitharama Raju,Cereals,Total,2024-25,0.76,2.08,2722.0
344990,Andhra Pradesh,Alluri Sitharama Raju,Maize,Kharif,2024-25,0.02,0.05,2535.0



Area = 0.01 and Production = 0.01: 16,123 matching records


,State,District,Crop,Season,Year,Area,Production,Yield
345022,Andhra Pradesh,Anakapalli,Bajra,Kharif,2024-25,0.01,0.01,1101.0
345023,Andhra Pradesh,Anakapalli,Bajra,Total,2024-25,0.01,0.01,1101.0
345037,Andhra Pradesh,Anakapalli,Nutri/Coarse Cereals,Kharif,2024-25,0.01,0.01,1251.0
345039,Andhra Pradesh,Anakapalli,Nutri/Coarse Cereals,Total,2024-25,0.01,0.01,1675.0
345049,Andhra Pradesh,Anakapalli,Shree Anna /Nutri Cereals,Kharif,2024-25,0.01,0.01,1131.0
345051,Andhra Pradesh,Anakapalli,Shree Anna /Nutri Cereals,Total,2024-25,0.01,0.01,1131.0
345136,Andhra Pradesh,Annamayya,Other Pulses,Kharif,2024-25,0.01,0.01,1290.0
345145,Andhra Pradesh,Annamayya,Shree Anna /Nutri Cereals,Kharif,2024-25,0.01,0.01,2179.0
345147,Andhra Pradesh,Annamayya,Shree Anna /Nutri Cereals,Total,2024-25,0.01,0.01,2177.0
345199,Andhra Pradesh,Bapatla,Tur,Kharif,2024-25,0.01,0.01,953.0


In [16]:
# Cell 16 — Test whether Area and Production use millions while Yield uses kg/ha.
# This is a diagnostic hypothesis; it does not transform or overwrite source data.

unit_test_df = apy_long_df.loc[
    (apy_long_df["Area"] > 0)
    & (apy_long_df["Production"] > 0)
    & (apy_long_df["Yield"] > 0)
].copy()

# If Area is in million hectares and Production in million tonnes,
# Production / Area is tonnes per hectare; multiply by 1,000 to get kg/ha.
unit_test_df["yield_implied_kg_per_ha"] = (
    unit_test_df["Production"] / unit_test_df["Area"] * 1000
)

unit_test_df["yield_to_implied_ratio"] = (
    unit_test_df["Yield"] / unit_test_df["yield_implied_kg_per_ha"]
)

unit_test_df["relative_deviation_from_implied_yield"] = (
    unit_test_df["Yield"] - unit_test_df["yield_implied_kg_per_ha"]
).abs() / unit_test_df["yield_implied_kg_per_ha"]

print(f"Records tested: {len(unit_test_df):,}")

print("\nAgreement with the proposed unit convention:")
display(
    pd.DataFrame({
        "criterion": [
            "Within 1% of implied yield",
            "Within 5% of implied yield",
            "Within 10% of implied yield",
            "More than 25% from implied yield",
        ],
        "records": [
            int((unit_test_df["relative_deviation_from_implied_yield"] <= 0.01).sum()),
            int((unit_test_df["relative_deviation_from_implied_yield"] <= 0.05).sum()),
            int((unit_test_df["relative_deviation_from_implied_yield"] <= 0.10).sum()),
            int((unit_test_df["relative_deviation_from_implied_yield"] > 0.25).sum()),
        ],
    }).assign(
        percent=lambda df: (
            100 * df["records"] / len(unit_test_df)
        ).round(2)
    )
)

print("\nRepresentative records:")
display(
    unit_test_df[
        [
            "State", "District", "Crop", "Season", "Year",
            "Area", "Production", "Yield",
            "yield_implied_kg_per_ha",
            "relative_deviation_from_implied_yield",
        ]
    ].head(10).round(3)
)

print("\nNo source values were modified.")

Records tested: 186,587

Agreement with the proposed unit convention:


,criterion,records,percent
0,Within 1% of implied yield,82755,44.35
1,Within 5% of implied yield,125862,67.45
2,Within 10% of implied yield,144849,77.63
3,More than 25% from implied yield,18218,9.76



Representative records:


,State,District,Crop,Season,Year,Area,Production,Yield,yield_implied_kg_per_ha,relative_deviation_from_implied_yield
19,Andaman And Nicobar Islands,North And Middle Andaman,Cereals,Kharif,2013-14,0.08,0.15,2010.0,1875.0,0.072
21,Andaman And Nicobar Islands,North And Middle Andaman,Cereals,Total,2013-14,0.08,0.16,2013.0,2000.0,0.006
32,Andaman And Nicobar Islands,North And Middle Andaman,Rice,Kharif,2013-14,0.08,0.15,2010.0,1875.0,0.072
33,Andaman And Nicobar Islands,North And Middle Andaman,Rice,Total,2013-14,0.08,0.15,2010.0,1875.0,0.072
34,Andaman And Nicobar Islands,North And Middle Andaman,Total Food Grains,Kharif,2013-14,0.08,0.15,2010.0,1875.0,0.072
35,Andaman And Nicobar Islands,North And Middle Andaman,Total Food Grains,Rabi,2013-14,0.01,0.03,2698.0,3000.0,0.101
36,Andaman And Nicobar Islands,North And Middle Andaman,Total Food Grains,Total,2013-14,0.09,0.18,2093.0,2000.0,0.046
37,Andaman And Nicobar Islands,North And Middle Andaman,Total Pulses,Total,2013-14,0.01,0.03,2726.0,3000.0,0.091
60,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Total,2013-14,0.01,0.02,2973.0,2000.0,0.486
147,Andhra Pradesh,Ananthapuramu,Bajra,Kharif,2013-14,0.01,0.01,921.0,1000.0,0.079



No source values were modified.


In [17]:
# Cell 17 — Check whether rounded, small-area measurements explain yield discrepancies.
# This is diagnostic only; source values remain unchanged.

unit_test_df["area_size_group"] = pd.cut(
    unit_test_df["Area"],
    bins=[-float("inf"), 0.01, 0.05, 0.10, 0.50, float("inf")],
    labels=[
        "≤ 0.01",
        "0.01–0.05",
        "0.05–0.10",
        "0.10–0.50",
        "> 0.50",
    ],
    include_lowest=True,
)

area_agreement = (
    unit_test_df.groupby("area_size_group", observed=False)
    .agg(
        records=("Yield", "size"),
        median_area=("Area", "median"),
        median_relative_deviation=(
            "relative_deviation_from_implied_yield", "median"
        ),
        within_5_percent=(
            "relative_deviation_from_implied_yield",
            lambda s: (s <= 0.05).mean() * 100,
        ),
        over_25_percent=(
            "relative_deviation_from_implied_yield",
            lambda s: (s > 0.25).mean() * 100,
        ),
    )
    .reset_index()
)

area_agreement[
    ["median_relative_deviation", "within_5_percent", "over_25_percent"]
] = area_agreement[
    ["median_relative_deviation", "within_5_percent", "over_25_percent"]
].round(2)

display(area_agreement)

print("\nInterpretation caution:")
print(
    "A relationship between small Area values and larger discrepancies "
    "would support a rounding-sensitivity explanation, not prove the "
    "official units or justify automatic correction."
)

,area_size_group,records,median_area,median_relative_deviation,within_5_percent,over_25_percent
0,≤ 0.01,26982,0.01,0.23,12.72,46.57
1,0.01–0.05,40059,0.03,0.10,28.80,13.66
2,0.05–0.10,19979,0.08,0.04,65.99,0.74
3,0.10–0.50,45141,0.23,0.01,95.91,0.07
4,> 0.50,54426,1.21,0.00,99.98,0.00



Interpretation caution:
A relationship between small Area values and larger discrepancies would support a rounding-sensitivity explanation, not prove the official units or justify automatic correction.


In [18]:
# Cell 18 — Verify the area-group audit table before interpreting it.

print("Columns:", area_agreement.columns.tolist())

display(
    area_agreement.to_string(index=False)
)

print("\nSanity checks:")
print(
    "Rows in area-group summary:",
    int(area_agreement["records"].sum())
)
print(
    "Rows in unit test:",
    len(unit_test_df)
)
print(
    "All group counts reconcile:",
    int(area_agreement["records"].sum()) == len(unit_test_df)
)

Columns: ['area_size_group', 'records', 'median_area', 'median_relative_deviation', 'within_5_percent', 'over_25_percent']


'area_size_group  records  median_area  median_relative_deviation  within_5_percent  over_25_percent\n         ≤ 0.01    26982         0.01                       0.23             12.72            46.57\n      0.01–0.05    40059         0.03                       0.10             28.80            13.66\n      0.05–0.10    19979         0.08                       0.04             65.99             0.74\n      0.10–0.50    45141         0.23                       0.01             95.91             0.07\n         > 0.50    54426         1.21                       0.00             99.98             0.00'


Sanity checks:
Rows in area-group summary: 186587
Rows in unit test: 186587
All group counts reconcile: True


In [19]:
# Cell 19 — Save the measurement consistency audit and its limitations.

from pathlib import Path

audit_dir = Path("data/processed/weather_regional")
audit_dir.mkdir(parents=True, exist_ok=True)

# Preserve the detailed area-group diagnostics.
area_agreement.to_csv(
    audit_dir / "yield_consistency_by_area_size.csv",
    index=False
)

# Summarize the comparison without changing or filtering source measurements.
measurement_audit_summary = pd.DataFrame([
    {
        "metric": "records_tested",
        "value": len(unit_test_df),
        "interpretation": "Positive finite Area, Production, and Yield",
    },
    {
        "metric": "within_1_percent",
        "value": int(
            (unit_test_df["relative_deviation_from_implied_yield"] <= 0.01).sum()
        ),
        "interpretation": "Reported Yield versus proposed implied yield",
    },
    {
        "metric": "within_5_percent",
        "value": int(
            (unit_test_df["relative_deviation_from_implied_yield"] <= 0.05).sum()
        ),
        "interpretation": "Reported Yield versus proposed implied yield",
    },
    {
        "metric": "within_10_percent",
        "value": int(
            (unit_test_df["relative_deviation_from_implied_yield"] <= 0.10).sum()
        ),
        "interpretation": "Reported Yield versus proposed implied yield",
    },
    {
        "metric": "more_than_25_percent_deviation",
        "value": int(
            (unit_test_df["relative_deviation_from_implied_yield"] > 0.25).sum()
        ),
        "interpretation": "Requires investigation; not automatically invalid",
    },
])

measurement_audit_summary.to_csv(
    audit_dir / "yield_measurement_consistency_summary.csv",
    index=False
)

print("Saved:")
print(audit_dir / "yield_consistency_by_area_size.csv")
print(audit_dir / "yield_measurement_consistency_summary.csv")

print("\nAudit status:")
print("- Source measurements modified: No")
print("- Unit conversion applied: No")
print("- Official unit definition verified for this extract: No")
print("- Small-area rounding sensitivity: Supported by observed association")
print("- Automatic exclusion based on ratio deviation: No")

display(measurement_audit_summary)

Saved:
data\processed\weather_regional\yield_consistency_by_area_size.csv
data\processed\weather_regional\yield_measurement_consistency_summary.csv

Audit status:
- Source measurements modified: No
- Unit conversion applied: No
- Official unit definition verified for this extract: No
- Small-area rounding sensitivity: Supported by observed association
- Automatic exclusion based on ratio deviation: No


,metric,value,interpretation
0,records_tested,186587,"Positive finite Area, Production, and Yield"
1,within_1_percent,82755,Reported Yield versus proposed implied yield
2,within_5_percent,125862,Reported Yield versus proposed implied yield
3,within_10_percent,144849,Reported Yield versus proposed implied yield
4,more_than_25_percent_deviation,18218,Requires investigation; not automatically invalid


In [20]:
# Cell 20 — Create the candidate modelling cohort without changing source data.
# Exclude aggregate "Total" seasons and invalid targets; retain all audit flags.

model_df = feature_df.loc[
    feature_df["FE_BasicTargetEligible"]
].copy()

# Standardize target and time-field types for downstream modelling.
model_df["Yield"] = pd.to_numeric(model_df["Yield"], errors="coerce")
model_df["Year"] = model_df["Year"].astype(str).str.strip()

# Identify duplicate prediction keys without silently dropping records.
prediction_key = ["State", "District", "Crop", "Season", "Year"]

model_df["FE_DuplicatePredictionKey"] = model_df.duplicated(
    subset=prediction_key,
    keep=False
)

print("Candidate modelling cohort")
print("-" * 40)
print(f"Rows: {len(model_df):,}")
print(f"Columns: {len(model_df.columns):,}")
print(f"Unique years: {model_df['Year'].nunique():,}")
print(f"Unique crops: {model_df['Crop'].nunique():,}")
print(f"Unique seasons: {model_df['Season'].nunique():,}")
print(f"Missing targets: {model_df['Yield'].isna().sum():,}")
print(
    "Duplicate prediction-key rows:",
    f"{model_df['FE_DuplicatePredictionKey'].sum():,}"
)

print("\nRows by season:")
display(
    model_df["Season"]
    .value_counts(dropna=False)
    .rename_axis("Season")
    .reset_index(name="rows")
)

print("\nDuplicate-key examples:")
display(
    model_df.loc[
        model_df["FE_DuplicatePredictionKey"],
        prediction_key + ["Area", "Production", "Yield"]
    ].head(10)
)

print("\nSource data unchanged:", len(feature_df) == len(integrated_df))

Candidate modelling cohort
----------------------------------------
Rows: 166,813
Columns: 46
Unique years: 12
Unique crops: 18
Unique seasons: 3
Missing targets: 0
Duplicate prediction-key rows: 0

Rows by season:


,Season,rows
0,Kharif,79381
1,Rabi,64025
2,Summer,23407



Duplicate-key examples:


,State,District,Crop,Season,Year,Area,Production,Yield



Source data unchanged: True


In [21]:
# Cell 21 — Audit target coverage and distribution by year.
# This is diagnostic; it does not filter or modify the modelling cohort.

model_df["Yield"] = pd.to_numeric(model_df["Yield"], errors="coerce")

yearly_target_audit = (
    model_df.groupby("Year", dropna=False)
    .agg(
        records=("Yield", "size"),
        unique_states=("State", "nunique"),
        unique_districts=("District", "nunique"),
        unique_crops=("Crop", "nunique"),
        median_yield=("Yield", "median"),
        mean_yield=("Yield", "mean"),
        p90_yield=("Yield", lambda s: s.quantile(0.90)),
        max_yield=("Yield", "max"),
    )
    .reset_index()
)

yearly_target_audit[
    ["median_yield", "mean_yield", "p90_yield", "max_yield"]
] = yearly_target_audit[
    ["median_yield", "mean_yield", "p90_yield", "max_yield"]
].round(2)

display(yearly_target_audit)

print("\nCohort reconciliation:")
print("Rows in model_df:", f"{len(model_df):,}")
print("Rows in yearly audit:", f"{yearly_target_audit['records'].sum():,}")
print(
    "All rows accounted for:",
    int(yearly_target_audit["records"].sum()) == len(model_df)
)

print("\nYield quantiles for the full cohort:")
display(
    model_df["Yield"]
    .quantile([0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
    .rename("Yield")
    .to_frame()
)

,Year,records,unique_states,unique_districts,unique_crops,median_yield,mean_yield,p90_yield,max_yield
0,2013-14,12239,34,635,18,1400.0,1744.48,3431.2,11375.0
1,2014-15,12423,33,652,18,1456.0,1719.12,3249.0,11292.0
2,2015-16,12697,34,657,18,1303.0,1640.76,3205.0,11978.0
3,2016-17,13222,34,683,18,1473.0,1784.89,3423.0,19317.0
4,2017-18,13869,34,683,18,1578.0,1921.26,3715.4,16609.0
5,2018-19,13870,34,687,18,1478.0,1846.53,3597.0,22000.0
6,2019-20,14382,33,693,18,1577.0,1932.23,3748.0,27560.0
7,2020-21,14585,33,703,18,1659.0,1984.54,3716.6,21300.0
8,2021-22,14341,34,706,18,1627.0,1950.97,3686.0,103571.0
9,2022-23,14994,34,732,18,1704.0,2059.48,3835.0,44815.0



Cohort reconciliation:
Rows in model_df: 166,813
Rows in yearly audit: 166,813
All rows accounted for: True

Yield quantiles for the full cohort:


,Yield
0.00,1.00
0.01,200.00
0.05,375.00
0.25,866.00
0.50,1560.00
0.75,2642.00
0.90,3650.00
0.95,4342.00
0.99,6716.88
1.00,385540.00


In [22]:
# Cell 22 — Verify the yearly audit table's actual values and column alignment.

print("Columns:", yearly_target_audit.columns.tolist())

print("\nYearly audit, formatted:")
print(yearly_target_audit.to_string(index=False))

print("\nCheck key invariants:")
print(
    "Record counts reconcile:",
    int(yearly_target_audit["records"].sum()) == len(model_df)
)
print(
    "Maximum yield equals the cohort maximum:",
    yearly_target_audit["max_yield"].max() == model_df["Yield"].max()
)
print(
    "Years are unique:",
    yearly_target_audit["Year"].is_unique
)

print("\nHighest-yield records in the cohort:")
display(
    model_df.nlargest(10, "Yield")[
        ["State", "District", "Crop", "Season", "Year", "Area", "Production", "Yield"]
    ]
)

Columns: ['Year', 'records', 'unique_states', 'unique_districts', 'unique_crops', 'median_yield', 'mean_yield', 'p90_yield', 'max_yield']

Yearly audit, formatted:
   Year  records  unique_states  unique_districts  unique_crops  median_yield  mean_yield  p90_yield  max_yield
2013-14    12239             34               635            18        1400.0     1744.48     3431.2    11375.0
2014-15    12423             33               652            18        1456.0     1719.12     3249.0    11292.0
2015-16    12697             34               657            18        1303.0     1640.76     3205.0    11978.0
2016-17    13222             34               683            18        1473.0     1784.89     3423.0    19317.0
2017-18    13869             34               683            18        1578.0     1921.26     3715.4    16609.0
2018-19    13870             34               687            18        1478.0     1846.53     3597.0    22000.0
2019-20    14382             33               693   

,State,District,Crop,Season,Year,Area,Production,Yield
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,0.00,0.00,385540.0
313619,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2023-24,0.00,0.00,161579.0
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,0.00,0.00,105588.0
265536,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,2021-22,0.00,0.01,103571.0
264973,Madhya Pradesh,Harda,Other Pulses,Rabi,2021-22,0.00,0.01,98889.0
288718,Goa,North Goa,Cereals,Kharif,2022-23,0.01,0.47,44815.0
288727,Goa,North Goa,Rice,Kharif,2022-23,0.01,0.47,44815.0
288732,Goa,North Goa,Total Food Grains,Kharif,2022-23,0.01,0.47,44815.0
288736,Goa,South Goa,Cereals,Kharif,2022-23,0.01,0.47,39221.0
288745,Goa,South Goa,Rice,Kharif,2022-23,0.01,0.47,39221.0


In [23]:
# Cell 23 — Profile extreme yields and their associated Area/Production values.
# Thresholds identify records for review; they do not define automatic exclusions.

yield_thresholds = [6551, 10000, 20000]

extreme_yield_audit = []

for threshold in yield_thresholds:
    subset = model_df.loc[model_df["Yield"] > threshold].copy()

    extreme_yield_audit.append({
        "yield_threshold_exclusive": threshold,
        "records": len(subset),
        "area_zero": int((subset["Area"] == 0).sum()),
        "production_zero": int((subset["Production"] == 0).sum()),
        "both_area_and_production_zero": int(
            ((subset["Area"] == 0) & (subset["Production"] == 0)).sum()
        ),
        "area_or_production_nonpositive": int(
            ((subset["Area"] <= 0) | (subset["Production"] <= 0)).sum()
        ),
        "share_both_zero_percent": round(
            100 * ((subset["Area"] == 0) & (subset["Production"] == 0)).mean(), 2
        ) if len(subset) else 0.0,
    })

extreme_yield_audit = pd.DataFrame(extreme_yield_audit)
display(extreme_yield_audit)

print("\nHighest-yield records, including source quality flags:")
quality_columns = [
    column for column in model_df.columns
    if column.startswith("QC_") or column.startswith("FE_")
]

display(
    model_df.nlargest(15, "Yield")[
        [
            "State", "District", "Crop", "Season", "Year",
            "Area", "Production", "Yield"
        ] + quality_columns
    ]
)

print("\n2024-25 maximum-yield records:")
display(
    model_df.loc[
        (model_df["Year"] == "2024-25")
        & (model_df["Yield"] == model_df.loc[
            model_df["Year"] == "2024-25", "Yield"
        ].max()),
        ["State", "District", "Crop", "Season", "Year",
         "Area", "Production", "Yield"]
    ]
)

,yield_threshold_exclusive,records,area_zero,production_zero,both_area_and_production_zero,area_or_production_nonpositive,share_both_zero_percent
0,6551,1831,487,228,228,487,12.45
1,10000,162,61,24,24,61,14.81
2,20000,22,12,8,8,12,36.36



Highest-yield records, including source quality flags:


,State,District,Crop,Season,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,...,QC_High_Annual_Precipitation,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance,FE_ValidPositiveYield,FE_IsAggregateTotal,FE_InvalidAreaProductionForYield,FE_ExtremeYieldReview,FE_BasicTargetEligible,FE_ConsistentMeasurementEligible,FE_DuplicatePredictionKey
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,0.00,0.00,385540.0,False,False,...,False,False,False,True,False,True,True,True,False,False
313619,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2023-24,0.00,0.00,161579.0,False,False,...,False,False,False,True,False,True,True,True,False,False
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,0.00,0.00,105588.0,False,False,...,False,False,False,True,False,True,True,True,False,False
265536,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,2021-22,0.00,0.01,103571.0,True,False,...,False,False,False,True,False,True,True,True,False,False
264973,Madhya Pradesh,Harda,Other Pulses,Rabi,2021-22,0.00,0.01,98889.0,True,False,...,False,False,True,True,False,True,True,True,False,False
288718,Goa,North Goa,Cereals,Kharif,2022-23,0.01,0.47,44815.0,False,False,...,False,False,True,True,False,False,True,True,True,False
288727,Goa,North Goa,Rice,Kharif,2022-23,0.01,0.47,44815.0,False,False,...,False,False,True,True,False,False,True,True,True,False
288732,Goa,North Goa,Total Food Grains,Kharif,2022-23,0.01,0.47,44815.0,False,False,...,False,False,True,True,False,False,True,True,True,False
288736,Goa,South Goa,Cereals,Kharif,2022-23,0.01,0.47,39221.0,False,False,...,False,False,True,True,False,False,True,True,True,False
288745,Goa,South Goa,Rice,Kharif,2022-23,0.01,0.47,39221.0,False,False,...,False,False,True,True,False,False,True,True,True,False



2024-25 maximum-yield records:


,State,District,Crop,Season,Year,Area,Production,Yield
366307,Rajasthan,Bikaner,Other Pulses,Rabi,2024-25,0.0,0.0,15033.0


In [24]:
# Cell 24 — Define transparent target-quality cohorts for model evaluation.
# These cohorts distinguish basic eligibility from measurement consistency.

model_df["FE_TargetQualityCohort"] = "REVIEW_INVALID_TARGET"

valid_target = (
    np.isfinite(model_df["Yield"])
    & (model_df["Yield"] > 0)
)

positive_area_production = (
    np.isfinite(model_df["Area"])
    & np.isfinite(model_df["Production"])
    & (model_df["Area"] > 0)
    & (model_df["Production"] > 0)
)

model_df.loc[
    valid_target & ~positive_area_production,
    "FE_TargetQualityCohort"
] = "POSITIVE_YIELD_MEASUREMENT_REVIEW"

model_df.loc[
    valid_target & positive_area_production,
    "FE_TargetQualityCohort"
] = "POSITIVE_YIELD_MEASUREMENTS_POSITIVE"

cohort_summary = (
    model_df.groupby("FE_TargetQualityCohort", dropna=False)
    .agg(
        records=("Yield", "size"),
        median_yield=("Yield", "median"),
        p95_yield=("Yield", lambda s: s.quantile(0.95)),
        maximum_yield=("Yield", "max"),
        area_zero=("Area", lambda s: int((s == 0).sum())),
        production_zero=("Production", lambda s: int((s == 0).sum())),
    )
    .reset_index()
)

display(cohort_summary)

print("\nReconciliation:")
print("Total cohort rows:", f"{len(model_df):,}")
print("Quality cohorts sum:", f"{cohort_summary['records'].sum():,}")
print(
    "All rows accounted for:",
    int(cohort_summary["records"].sum()) == len(model_df)
)

print("\nSource dataset modified: No")
print("Rows deleted: No")
print("Yield values transformed: No")

,FE_TargetQualityCohort,records,median_yield,p95_yield,maximum_yield,area_zero,production_zero
0,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,107267,1912.0,4561.0,44815.0,0,0
1,POSITIVE_YIELD_MEASUREMENT_REVIEW,59546,1000.0,3667.0,385540.0,55803,52947



Reconciliation:
Total cohort rows: 166,813
Quality cohorts sum: 166,813
All rows accounted for: True

Source dataset modified: No
Rows deleted: No
Yield values transformed: No


In [25]:
# Cell 25 — Verify target-quality cohort values and reconcile them to model_df.

print("Columns:", cohort_summary.columns.tolist())
print("\nFull cohort summary:")
print(cohort_summary.to_string(index=False))

print("\nDirect maximum-yield checks:")
print("Overall maximum:", model_df["Yield"].max())

for cohort, group in model_df.groupby("FE_TargetQualityCohort"):
    print(f"{cohort}: rows={len(group):,}, max_yield={group['Yield'].max()}")

print("\nHighest-yield review-cohort records:")
display(
    model_df.loc[
        model_df["FE_TargetQualityCohort"]
        == "POSITIVE_YIELD_MEASUREMENT_REVIEW"
    ].nlargest(10, "Yield")[
        [
            "State", "District", "Crop", "Season", "Year",
            "Area", "Production", "Yield", "FE_TargetQualityCohort"
        ]
    ]
)

print(
    "\nCohort counts reconcile:",
    int(cohort_summary["records"].sum()) == len(model_df)
)

Columns: ['FE_TargetQualityCohort', 'records', 'median_yield', 'p95_yield', 'maximum_yield', 'area_zero', 'production_zero']

Full cohort summary:
              FE_TargetQualityCohort  records  median_yield  p95_yield  maximum_yield  area_zero  production_zero
POSITIVE_YIELD_MEASUREMENTS_POSITIVE   107267        1912.0     4561.0        44815.0          0                0
   POSITIVE_YIELD_MEASUREMENT_REVIEW    59546        1000.0     3667.0       385540.0      55803            52947

Direct maximum-yield checks:
Overall maximum: 385540.0
POSITIVE_YIELD_MEASUREMENTS_POSITIVE: rows=107,267, max_yield=44815.0
POSITIVE_YIELD_MEASUREMENT_REVIEW: rows=59,546, max_yield=385540.0

Highest-yield review-cohort records:


,State,District,Crop,Season,Year,Area,Production,Yield,FE_TargetQualityCohort
313624,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,2023-24,0.0,0.00,385540.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
313619,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2023-24,0.0,0.00,161579.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
313609,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,2023-24,0.0,0.00,105588.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
265536,Madhya Pradesh,Narmadapuram,Other Pulses,Rabi,2021-22,0.0,0.01,103571.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
264973,Madhya Pradesh,Harda,Other Pulses,Rabi,2021-22,0.0,0.01,98889.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
196915,Haryana,Yamunanagar,Other Pulses,Kharif,2019-20,0.0,0.06,27560.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
291610,Jammu And Kashmir,Jammu,Other Pulses,Rabi,2022-23,0.0,0.00,26360.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
213450,Telangana,Mulugu,Tur,Kharif,2019-20,0.0,0.01,22846.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
182809,The Dadra And Nagar Haveli And Daman And Diu,Dadra And Nagar Haveli,Jowar,Kharif,2018-19,0.0,0.00,22000.0,POSITIVE_YIELD_MEASUREMENT_REVIEW
239488,Odisha,Nuapada,Nutri/Coarse Cereals,Summer,2020-21,0.0,0.00,21300.0,POSITIVE_YIELD_MEASUREMENT_REVIEW



Cohort counts reconcile: True


In [26]:
# Cell 26 — Compare target-quality cohorts across years.
# No rows are deleted and no target values are modified.

cohort_by_year = (
    model_df.groupby(["Year", "FE_TargetQualityCohort"], observed=True)
    .agg(
        records=("Yield", "size"),
        median_yield=("Yield", "median"),
        p95_yield=("Yield", lambda s: s.quantile(0.95)),
    )
    .reset_index()
)

year_cohort_counts = cohort_by_year.pivot(
    index="Year",
    columns="FE_TargetQualityCohort",
    values="records"
).fillna(0)

year_cohort_counts["total_records"] = year_cohort_counts.sum(axis=1)

review_cohort_name = "POSITIVE_YIELD_MEASUREMENT_REVIEW"
positive_cohort_name = "POSITIVE_YIELD_MEASUREMENTS_POSITIVE"

year_cohort_counts["review_share_percent"] = (
    100 * year_cohort_counts[review_cohort_name]
    / year_cohort_counts["total_records"]
).round(2)

print("Record counts and review-cohort share by year:")
display(year_cohort_counts.reset_index())

print("\nTarget summaries by year and cohort:")
display(cohort_by_year)

print("\nOverall cohort share:")
display(
    model_df["FE_TargetQualityCohort"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percent")
    .to_frame()
)

Record counts and review-cohort share by year:


FE_TargetQualityCohort,Year,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,POSITIVE_YIELD_MEASUREMENT_REVIEW,total_records,review_share_percent
0,2013-14,8205,4034,12239,32.96
1,2014-15,8544,3879,12423,31.22
2,2015-16,8475,4222,12697,33.25
3,2016-17,8791,4431,13222,33.51
4,2017-18,9177,4692,13869,33.83
5,2018-19,8836,5034,13870,36.29
6,2019-20,9086,5296,14382,36.82
7,2020-21,9127,5458,14585,37.42
8,2021-22,8973,5368,14341,37.43
9,2022-23,9337,5657,14994,37.73



Target summaries by year and cohort:


,Year,FE_TargetQualityCohort,records,median_yield,p95_yield
0,2013-14,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,8205,1683.0,4513.20
1,2013-14,POSITIVE_YIELD_MEASUREMENT_REVIEW,4034,965.0,3485.00
2,2014-15,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,8544,1687.5,4069.85
3,2014-15,POSITIVE_YIELD_MEASUREMENT_REVIEW,3879,989.0,3224.80
4,2015-16,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,8475,1591.0,4104.70
5,2015-16,POSITIVE_YIELD_MEASUREMENT_REVIEW,4222,912.5,3060.00
6,2016-17,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,8791,1806.0,4357.00
7,2016-17,POSITIVE_YIELD_MEASUREMENT_REVIEW,4431,971.0,3134.00
8,2017-18,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,9177,1919.0,4658.20
9,2017-18,POSITIVE_YIELD_MEASUREMENT_REVIEW,4692,1000.0,3729.65



Overall cohort share:


,percent
FE_TargetQualityCohort,
POSITIVE_YIELD_MEASUREMENTS_POSITIVE,64.3
POSITIVE_YIELD_MEASUREMENT_REVIEW,35.7


In [27]:
# Cell 27 — Define modelling cohorts and candidate feature groups.
# Preserve original values; this cell only creates modelling views and metadata.

import numpy as np
import pandas as pd

# Primary candidate: positive target with positive Area and Production.
primary_df = model_df.loc[
    model_df["FE_TargetQualityCohort"]
    == "POSITIVE_YIELD_MEASUREMENTS_POSITIVE"
].copy()

# Sensitivity cohort: all positive finite targets, including measurement-review rows.
sensitivity_df = model_df.copy()

# Extract the starting year from labels such as "2013-14".
# Keep the original Year label unchanged.
for cohort_df in (primary_df, sensitivity_df):
    cohort_df["FE_YearStart"] = pd.to_numeric(
        cohort_df["Year"].astype(str).str.extract(r"^(\d{4})", expand=False),
        errors="coerce"
    ).astype("Int64")

# Core predictors: available before observing the target.
core_categorical_features = ["Crop", "Season", "Crop_Category"]
core_numeric_features = ["FE_YearStart"]

# Weather is an exploratory feature group because seasonal alignment
# and source-unit verification are still unresolved.
weather_features = [
    "tmin_mean_c",
    "tmin_min_c",
    "tmax_mean_c",
    "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day",
    "tmin_valid_days",
    "tmax_valid_days",
    "precipitation_valid_days",
]

# Keep only weather columns actually present in the modelling data.
weather_features = [
    col for col in weather_features if col in primary_df.columns
]

core_features = core_categorical_features + core_numeric_features
core_plus_weather_features = core_features + weather_features

print("Cohort sizes:")
print(f"Primary candidate:     {len(primary_df):,}")
print(f"Sensitivity cohort:    {len(sensitivity_df):,}")

print("\nFeature groups:")
print("Core categorical:", core_categorical_features)
print("Core numeric:", core_numeric_features)
print("Exploratory weather:", weather_features)

print("\nYear parsing checks:")
print("Primary missing start year:", int(primary_df["FE_YearStart"].isna().sum()))
print("Sensitivity missing start year:", int(sensitivity_df["FE_YearStart"].isna().sum()))
print("Primary year range:",
      primary_df["FE_YearStart"].min(),
      "to",
      primary_df["FE_YearStart"].max())

print("\nMissingness in candidate predictors (primary cohort):")
missingness = (
    primary_df[core_features + weather_features]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .rename("missing_percent")
    .to_frame()
)
display(missingness)

print("\nLeakage guard:")
for forbidden in ["Area", "Production", "Yield", "State", "District"]:
    print(f"{forbidden}: excluded from candidate feature lists = "
          f"{forbidden not in core_plus_weather_features}")

Cohort sizes:
Primary candidate:     107,267
Sensitivity cohort:    166,813

Feature groups:
Core categorical: ['Crop', 'Season', 'Crop_Category']
Core numeric: ['FE_YearStart']
Exploratory weather: ['tmin_mean_c', 'tmin_min_c', 'tmax_mean_c', 'tmax_max_c', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'tmin_valid_days', 'tmax_valid_days', 'precipitation_valid_days']

Year parsing checks:
Primary missing start year: 0
Sensitivity missing start year: 0
Primary year range: 2013 to 2024

Missingness in candidate predictors (primary cohort):


,missing_percent
tmin_mean_c,13.72
tmin_min_c,13.72
precipitation_mean_mm_day,13.72
precipitation_total_mm,13.72
tmax_max_c,13.72
tmax_mean_c,13.72
precipitation_max_mm_day,13.72
tmin_valid_days,13.72
tmax_valid_days,13.72
precipitation_valid_days,13.72



Leakage guard:
Area: excluded from candidate feature lists = True
Production: excluded from candidate feature lists = True
Yield: excluded from candidate feature lists = True
State: excluded from candidate feature lists = True
District: excluded from candidate feature lists = True


In [28]:
# Cell 28 — Build candidate feature matrices without target leakage.
# No imputation, row deletion, or source-data modification occurs here.

target_col = "Yield"

# Keep only the explicitly approved candidate predictors.
X_primary_core = primary_df[core_features].copy()
X_primary_weather = primary_df[core_plus_weather_features].copy()

y_primary = primary_df[target_col].copy()

X_sensitivity_core = sensitivity_df[core_features].copy()
X_sensitivity_weather = sensitivity_df[core_plus_weather_features].copy()

y_sensitivity = sensitivity_df[target_col].copy()

# Check that feature and target indexes remain aligned.
assert X_primary_core.index.equals(y_primary.index)
assert X_primary_weather.index.equals(y_primary.index)
assert X_sensitivity_core.index.equals(y_sensitivity.index)
assert X_sensitivity_weather.index.equals(y_sensitivity.index)

# Validate target integrity.
assert np.isfinite(y_primary).all() and (y_primary > 0).all()
assert np.isfinite(y_sensitivity).all() and (y_sensitivity > 0).all()

# Explicitly check that target and measurement columns cannot enter predictors.
for name, X in {
    "primary_core": X_primary_core,
    "primary_weather": X_primary_weather,
    "sensitivity_core": X_sensitivity_core,
    "sensitivity_weather": X_sensitivity_weather,
}.items():
    forbidden_present = [
        col for col in ["Yield", "Area", "Production", "State", "District"]
        if col in X.columns
    ]
    assert not forbidden_present, f"{name} contains forbidden columns: {forbidden_present}"

print("Feature matrix shapes:")
print(f"Primary core:              {X_primary_core.shape}")
print(f"Primary core + weather:    {X_primary_weather.shape}")
print(f"Sensitivity core:          {X_sensitivity_core.shape}")
print(f"Sensitivity core + weather:{X_sensitivity_weather.shape}")

print("\nTarget summary — primary candidate:")
display(y_primary.describe().to_frame("Yield"))

print("\nTarget summary — sensitivity cohort:")
display(y_sensitivity.describe().to_frame("Yield"))

print("\nWeather missingness by year (primary candidate, %):")
weather_missing_by_year = (
    primary_df.groupby("Year")[weather_features]
    .apply(lambda group: group.isna().mean().mul(100))
    .round(2)
)
display(weather_missing_by_year)

print("\nWeather missingness by crop (primary candidate, %):")
weather_missing_by_crop = (
    primary_df.groupby("Crop")[weather_features]
    .apply(lambda group: group.isna().mean().mean() * 100)
    .round(2)
    .rename("mean_weather_missing_percent")
    .sort_values(ascending=False)
    .to_frame()
)
display(weather_missing_by_crop)

print("\nValidation passed: feature-target alignment, positive finite targets, and leakage guards.")

Feature matrix shapes:
Primary core:              (107267, 4)
Primary core + weather:    (107267, 14)
Sensitivity core:          (166813, 4)
Sensitivity core + weather:(166813, 14)

Target summary — primary candidate:


,Yield
count,107267.000000
mean,2163.625169
std,1422.898030
min,15.000000
25%,1111.000000
50%,1912.000000
75%,2912.000000
max,44815.000000



Target summary — sensitivity cohort:


,Yield
count,166813.000000
mean,1903.049606
std,1791.772907
min,1.000000
25%,866.000000
50%,1560.000000
75%,2642.000000
max,385540.000000



Weather missingness by year (primary candidate, %):


,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days
Year,,,,,,,,,,
2013-14,13.27,13.27,13.27,13.27,13.27,13.27,13.27,13.27,13.27,13.27
2014-15,13.39,13.39,13.39,13.39,13.39,13.39,13.39,13.39,13.39,13.39
2015-16,13.50,13.50,13.50,13.50,13.50,13.50,13.50,13.50,13.50,13.50
2016-17,13.64,13.64,13.64,13.64,13.64,13.64,13.64,13.64,13.64,13.64
2017-18,13.74,13.74,13.74,13.74,13.74,13.74,13.74,13.74,13.74,13.74
2018-19,13.84,13.84,13.84,13.84,13.84,13.84,13.84,13.84,13.84,13.84
2019-20,14.37,14.37,14.37,14.37,14.37,14.37,14.37,14.37,14.37,14.37
2020-21,14.20,14.20,14.20,14.20,14.20,14.20,14.20,14.20,14.20,14.20
2021-22,13.98,13.98,13.98,13.98,13.98,13.98,13.98,13.98,13.98,13.98



Weather missingness by crop (primary candidate, %):


,mean_weather_missing_percent
Crop,
Ragi,27.04
Rice,17.98
Other Pulses,15.27
Jowar,15.10
Cereals,15.05
Total Food Grains,14.66
Shree Anna /Nutri Cereals,14.04
Maize,13.31
Small Millets,13.02



Validation passed: feature-target alignment, positive finite targets, and leakage guards.


In [29]:
# Cell 29 — Diagnose weather missingness structure.
# Read-only analysis; no source values or modelling rows are changed.

weather_missing_mask = primary_df[weather_features].isna()

print("1. Number of missing weather fields per row:")
missing_fields_per_row = weather_missing_mask.sum(axis=1)
display(
    missing_fields_per_row.value_counts()
    .sort_index()
    .rename_axis("missing_weather_fields")
    .to_frame("records")
)

print("\n2. Rows with all weather fields present or all missing:")
all_weather_present = primary_df[weather_features].notna().all(axis=1)
all_weather_missing = primary_df[weather_features].isna().all(axis=1)
partially_missing = ~(all_weather_present | all_weather_missing)

weather_pattern_summary = pd.DataFrame({
    "pattern": [
        "All weather fields present",
        "All weather fields missing",
        "Partially missing weather fields",
    ],
    "records": [
        int(all_weather_present.sum()),
        int(all_weather_missing.sum()),
        int(partially_missing.sum()),
    ],
})
weather_pattern_summary["percent"] = (
    100 * weather_pattern_summary["records"] / len(primary_df)
).round(2)
display(weather_pattern_summary)

print("\n3. Complete-weather coverage by year:")
coverage_by_year = (
    primary_df.assign(FE_CompleteWeather=all_weather_present)
    .groupby("Year", observed=True)
    .agg(
        records=("Yield", "size"),
        complete_weather_records=("FE_CompleteWeather", "sum"),
    )
)
coverage_by_year["complete_weather_percent"] = (
    100 * coverage_by_year["complete_weather_records"]
    / coverage_by_year["records"]
).round(2)
display(coverage_by_year.reset_index())

print("\n4. Complete-weather coverage by crop:")
coverage_by_crop = (
    primary_df.assign(FE_CompleteWeather=all_weather_present)
    .groupby("Crop", observed=True)
    .agg(
        records=("Yield", "size"),
        complete_weather_records=("FE_CompleteWeather", "sum"),
    )
)
coverage_by_crop["complete_weather_percent"] = (
    100 * coverage_by_crop["complete_weather_records"]
    / coverage_by_crop["records"]
).round(2)
display(
    coverage_by_crop.sort_values("complete_weather_percent")
    .reset_index()
)

print("\n5. Pairwise agreement of weather missingness masks:")
mask_agreement = pd.DataFrame(index=weather_features, columns=weather_features, dtype=float)
for col_a in weather_features:
    for col_b in weather_features:
        mask_agreement.loc[col_a, col_b] = (
            weather_missing_mask[col_a] == weather_missing_mask[col_b]
        ).mean()

display(mask_agreement.round(3))

print("\nNo imputation or row filtering performed.")

1. Number of missing weather fields per row:


,records
missing_weather_fields,
0,92547
10,14720



2. Rows with all weather fields present or all missing:


,pattern,records,percent
0,All weather fields present,92547,86.28
1,All weather fields missing,14720,13.72
2,Partially missing weather fields,0,0.00



3. Complete-weather coverage by year:


,Year,records,complete_weather_records,complete_weather_percent
0,2013-14,8205,7116,86.73
1,2014-15,8544,7400,86.61
2,2015-16,8475,7331,86.50
3,2016-17,8791,7592,86.36
4,2017-18,9177,7916,86.26
5,2018-19,8836,7613,86.16
6,2019-20,9086,7780,85.63
7,2020-21,9127,7831,85.80
8,2021-22,8973,7719,86.02
9,2022-23,9337,8019,85.88



4. Complete-weather coverage by crop:


,Crop,records,complete_weather_records,complete_weather_percent
0,Ragi,1117,815,72.96
1,Rice,11143,9140,82.02
2,Other Pulses,3634,3079,84.73
3,Jowar,2907,2468,84.90
4,Cereals,17850,15163,84.95
5,Total Food Grains,18931,16155,85.34
6,Shree Anna /Nutri Cereals,5571,4789,85.96
7,Maize,8355,7243,86.69
8,Small Millets,1160,1009,86.98
9,Moong,3321,2892,87.08



5. Pairwise agreement of weather missingness masks:


,tmin_mean_c,tmin_min_c,tmax_mean_c,tmax_max_c,precipitation_total_mm,precipitation_mean_mm_day,precipitation_max_mm_day,tmin_valid_days,tmax_valid_days,precipitation_valid_days
tmin_mean_c,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
tmin_min_c,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
tmax_mean_c,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
tmax_max_c,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
precipitation_total_mm,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
precipitation_mean_mm_day,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
precipitation_max_mm_day,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
tmin_valid_days,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
tmax_valid_days,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
precipitation_valid_days,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0



No imputation or row filtering performed.


In [30]:
# Cell 30 — Diagnose missing weather by geography and year.
# Read-only audit; no rows are removed and no values are changed.

weather_availability = (
    primary_df.assign(FE_WeatherAvailable=all_weather_present)
    .groupby(["State", "District", "Year"], observed=True)
    .agg(
        records=("Yield", "size"),
        weather_available=("FE_WeatherAvailable", "sum"),
    )
    .reset_index()
)

weather_availability["weather_missing"] = (
    weather_availability["records"] - weather_availability["weather_available"]
)

weather_availability["weather_missing_percent"] = (
    100 * weather_availability["weather_missing"]
    / weather_availability["records"]
).round(2)

print("1. Overall district-year coverage pattern:")
district_year_summary = pd.Series({
    "district_year_groups": len(weather_availability),
    "groups_with_all_weather": int(
        (weather_availability["weather_missing"] == 0).sum()
    ),
    "groups_with_no_weather": int(
        (weather_availability["weather_available"] == 0).sum()
    ),
    "groups_with_partial_row_coverage": int(
        (
            (weather_availability["weather_available"] > 0)
            & (weather_availability["weather_missing"] > 0)
        ).sum()
    ),
})
display(district_year_summary.to_frame("value"))

print("\n2. District-year groups with no weather coverage:")
display(
    weather_availability.loc[
        weather_availability["weather_available"] == 0
    ]
    .sort_values(["Year", "State", "District"])
    .head(30)
)

print("\n3. Districts with the most missing-weather records:")
district_summary = (
    primary_df.assign(FE_WeatherAvailable=all_weather_present)
    .groupby(["State", "District"], observed=True)
    .agg(
        records=("Yield", "size"),
        weather_available=("FE_WeatherAvailable", "sum"),
    )
    .reset_index()
)
district_summary["weather_missing"] = (
    district_summary["records"] - district_summary["weather_available"]
)
district_summary["weather_missing_percent"] = (
    100 * district_summary["weather_missing"] / district_summary["records"]
).round(2)

display(
    district_summary.sort_values(
        ["weather_missing", "weather_missing_percent"],
        ascending=False
    ).head(25)
)

print("\n4. Missing-weather share by state:")
state_weather_summary = (
    primary_df.assign(FE_WeatherAvailable=all_weather_present)
    .groupby("State", observed=True)
    .agg(
        records=("Yield", "size"),
        weather_available=("FE_WeatherAvailable", "sum"),
    )
    .reset_index()
)
state_weather_summary["weather_missing_percent"] = (
    100
    * (state_weather_summary["records"] - state_weather_summary["weather_available"])
    / state_weather_summary["records"]
).round(2)

display(
    state_weather_summary.sort_values(
        "weather_missing_percent", ascending=False
    )
)

print("\nSource data remains unchanged.")

1. Overall district-year coverage pattern:


,value
district_year_groups,8230
groups_with_all_weather,7043
groups_with_no_weather,1187
groups_with_partial_row_coverage,0



2. District-year groups with no weather coverage:


,State,District,Year,records,weather_available,weather_missing,weather_missing_percent
193,Andhra Pradesh,Y.S.R. Kadapa,2013-14,22,0,22,100.0
369,Arunachal Pradesh,Papum Pare,2013-14,7,0,7,100.0
539,Assam,Darrang,2013-14,15,0,15,100.0
652,Assam,Kamrup,2013-14,13,0,13,100.0
753,Assam,Sivasagar,2013-14,7,0,7,100.0
1135,Bihar,Purbi Champaran,2013-14,19,0,19,100.0
1303,Chhattisgarh,Balodabazar-Bhatapara,2013-14,10,0,10,100.0
1315,Chhattisgarh,Balrampur-Ramanujganj,2013-14,16,0,16,100.0
1375,Chhattisgarh,Dakshin Bastar Dantewada,2013-14,7,0,7,100.0
1651,Gujarat,Ahmedabad,2013-14,18,0,18,100.0



3. Districts with the most missing-weather records:


,State,District,records,weather_available,weather_missing,weather_missing_percent
271,Karnataka,Ballari,318,0,318,100.0
272,Karnataka,Belagavi,309,0,309,100.0
282,Karnataka,Davanagere,285,0,285,100.0
292,Karnataka,Mysuru,281,0,281,100.0
286,Karnataka,Haveri,268,0,268,100.0
26,Andhra Pradesh,Y.S.R. Kadapa,263,0,263,100.0
379,Maharashtra,Chhatrapati Sambhajinagar,261,0,261,100.0
270,Karnataka,Bagalkote,260,0,260,100.0
733,West Bengal,Malda,254,0,254,100.0
277,Karnataka,Chamarajanagar,252,0,252,100.0



4. Missing-weather share by state:


,State,records,weather_available,weather_missing_percent
29,The Dadra And Nagar Haveli And Daman And Diu,128,0,100.00
14,Kerala,1042,0,100.00
13,Karnataka,6676,2279,65.86
33,West Bengal,3890,1533,60.59
15,Ladakh,100,50,50.00
11,Jammu And Kashmir,1366,958,29.87
22,Odisha,3745,2651,29.21
6,Chhattisgarh,3260,2670,18.10
28,Telangana,4069,3432,15.65
30,Tripura,981,842,14.17



Source data remains unchanged.


In [31]:
# Cell 31 — State-year weather coverage audit.
# Read-only analysis; no records are removed or modified.

state_year_coverage = (
    primary_df.assign(FE_WeatherAvailable=all_weather_present)
    .groupby(["State", "Year"], observed=True)
    .agg(
        records=("Yield", "size"),
        weather_available=("FE_WeatherAvailable", "sum"),
    )
    .reset_index()
)

state_year_coverage["weather_missing"] = (
    state_year_coverage["records"]
    - state_year_coverage["weather_available"]
)

state_year_coverage["weather_missing_percent"] = (
    100 * state_year_coverage["weather_missing"]
    / state_year_coverage["records"]
).round(2)

print("1. States with no weather coverage in any observed year:")
state_summary = (
    state_year_coverage.groupby("State", observed=True)
    .agg(
        total_records=("records", "sum"),
        weather_available=("weather_available", "sum"),
        years_observed=("Year", "nunique"),
        years_with_no_weather=(
            "weather_available",
            lambda s: int((s == 0).sum())
        ),
    )
    .reset_index()
)

state_summary["weather_missing_percent"] = (
    100
    * (state_summary["total_records"] - state_summary["weather_available"])
    / state_summary["total_records"]
).round(2)

display(
    state_summary.loc[
        state_summary["weather_available"] == 0
    ].sort_values("State")
)

print("\n2. State-year groups with no weather coverage:")
display(
    state_year_coverage.loc[
        state_year_coverage["weather_available"] == 0
    ].sort_values(["State", "Year"])
)

print("\n3. State-year groups with partial coverage:")
display(
    state_year_coverage.loc[
        (state_year_coverage["weather_available"] > 0)
        & (state_year_coverage["weather_missing"] > 0)
    ].sort_values(
        ["weather_missing_percent", "records"],
        ascending=[False, False]
    ).head(40)
)

print("\n4. Number of years with zero weather coverage per state:")
display(
    state_summary.sort_values(
        ["years_with_no_weather", "weather_missing_percent"],
        ascending=False
    ).head(20)
)

print("\nNo imputation, filtering, or source-data changes performed.")

1. States with no weather coverage in any observed year:


,State,total_records,weather_available,years_observed,years_with_no_weather,weather_missing_percent
14,Kerala,1042,0,12,12,100.0
29,The Dadra And Nagar Haveli And Daman And Diu,128,0,10,10,100.0



2. State-year groups with no weather coverage:


,State,Year,records,weather_available,weather_missing,weather_missing_percent
166,Kerala,2013-14,89,0,89,100.0
167,Kerala,2014-15,90,0,90,100.0
168,Kerala,2015-16,88,0,88,100.0
169,Kerala,2016-17,81,0,81,100.0
170,Kerala,2017-18,90,0,90,100.0
171,Kerala,2018-19,87,0,87,100.0
172,Kerala,2019-20,90,0,90,100.0
173,Kerala,2020-21,93,0,93,100.0
174,Kerala,2021-22,81,0,81,100.0
175,Kerala,2022-23,87,0,87,100.0



3. State-year groups with partial coverage:


,State,Year,records,weather_available,weather_missing,weather_missing_percent
162,Karnataka,2021-22,563,180,383,68.03
161,Karnataka,2020-21,589,191,398,67.57
160,Karnataka,2019-20,587,193,394,67.12
155,Karnataka,2014-15,565,187,378,66.90
156,Karnataka,2015-16,544,182,362,66.54
154,Karnataka,2013-14,547,184,363,66.36
159,Karnataka,2018-19,529,181,348,65.78
163,Karnataka,2022-23,597,208,389,65.16
157,Karnataka,2016-17,515,181,334,64.85
158,Karnataka,2017-18,543,191,352,64.83



4. Number of years with zero weather coverage per state:


,State,total_records,weather_available,years_observed,years_with_no_weather,weather_missing_percent
14,Kerala,1042,0,12,12,100.00
29,The Dadra And Nagar Haveli And Daman And Diu,128,0,10,10,100.00
15,Ladakh,100,50,10,2,50.00
13,Karnataka,6676,2279,12,0,65.86
33,West Bengal,3890,1533,12,0,60.59
11,Jammu And Kashmir,1366,958,12,0,29.87
22,Odisha,3745,2651,12,0,29.21
6,Chhattisgarh,3260,2670,12,0,18.10
28,Telangana,4069,3432,12,0,15.65
30,Tripura,981,842,12,0,14.17



No imputation, filtering, or source-data changes performed.


In [32]:
# Cell 32 — Create a chronological train/holdout split.
# No random split, imputation, or target modification.

train_end_year = 2022
holdout_start_year = 2023

primary_train = primary_df.loc[
    primary_df["FE_YearStart"] <= train_end_year
].copy()

primary_holdout = primary_df.loc[
    primary_df["FE_YearStart"] >= holdout_start_year
].copy()

sensitivity_train = sensitivity_df.loc[
    sensitivity_df["FE_YearStart"] <= train_end_year
].copy()

sensitivity_holdout = sensitivity_df.loc[
    sensitivity_df["FE_YearStart"] >= holdout_start_year
].copy()

# Confirm that the split is chronological and partitions each cohort.
assert primary_train["FE_YearStart"].max() < primary_holdout["FE_YearStart"].min()
assert sensitivity_train["FE_YearStart"].max() < sensitivity_holdout["FE_YearStart"].min()

assert len(primary_train) + len(primary_holdout) == len(primary_df)
assert len(sensitivity_train) + len(sensitivity_holdout) == len(sensitivity_df)

print("Primary candidate split:")
print(f"Train rows:   {len(primary_train):,}")
print(f"Holdout rows: {len(primary_holdout):,}")
print(
    f"Train years: {primary_train['Year'].min()} to "
    f"{primary_train['Year'].max()}"
)
print(
    f"Holdout years: {primary_holdout['Year'].min()} to "
    f"{primary_holdout['Year'].max()}"
)

print("\nSensitivity cohort split:")
print(f"Train rows:   {len(sensitivity_train):,}")
print(f"Holdout rows: {len(sensitivity_holdout):,}")

print("\nPrimary holdout cohort counts:")
display(
    primary_holdout.groupby(
        ["Year", "FE_TargetQualityCohort"], observed=True
    ).size().rename("records").reset_index()
)

print("\nWeather availability in primary train and holdout:")
for split_name, split_df in [
    ("train", primary_train),
    ("holdout", primary_holdout),
]:
    complete = split_df[weather_features].notna().all(axis=1)
    print(
        f"{split_name}: {int(complete.sum()):,}/{len(split_df):,} "
        f"rows with complete weather ({complete.mean() * 100:.2f}%)"
    )

print("\nChronological split validation passed.")

Primary candidate split:
Train rows:   88,551
Holdout rows: 18,716
Train years: 2013-14 to 2022-23
Holdout years: 2023-24 to 2024-25

Sensitivity cohort split:
Train rows:   136,622
Holdout rows: 30,191

Primary holdout cohort counts:


,Year,FE_TargetQualityCohort,records
0,2023-24,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,9353
1,2024-25,POSITIVE_YIELD_MEASUREMENTS_POSITIVE,9363



Weather availability in primary train and holdout:
train: 76,317/88,551 rows with complete weather (86.18%)
holdout: 16,230/18,716 rows with complete weather (86.72%)

Chronological split validation passed.


In [33]:
# Cell 33 — Final feature-engineering contract and audit.
# No model fitting, imputation, or source-data modification.

feature_engineering_contract = {
    "notebook": "15_feature_engineering.ipynb",
    "target": "Yield",
    "prediction_unit": "district_crop_season_year",
    "primary_cohort": "POSITIVE_YIELD_MEASUREMENTS_POSITIVE",
    "sensitivity_cohort": "All positive finite non-aggregate yields",
    "train_years": sorted(primary_train["FE_YearStart"].unique().tolist()),
    "holdout_years": sorted(primary_holdout["FE_YearStart"].unique().tolist()),
    "split_strategy": "Chronological holdout; no random-only split",
    "core_categorical_features": core_categorical_features,
    "core_numeric_features": core_numeric_features,
    "exploratory_weather_features": weather_features,
    "weather_alignment": "STARTING_CALENDAR_YEAR_PROXY",
    "full_season_weather_alignment_verified": False,
    "yield_area_production_units_verified": False,
    "causal_claims_permitted": False,
}

feature_engineering_audit = pd.DataFrame([
    {
        "cohort": "primary",
        "train_rows": len(primary_train),
        "holdout_rows": len(primary_holdout),
        "train_weather_complete_rows": int(
            primary_train[weather_features].notna().all(axis=1).sum()
        ),
        "holdout_weather_complete_rows": int(
            primary_holdout[weather_features].notna().all(axis=1).sum()
        ),
    },
    {
        "cohort": "sensitivity",
        "train_rows": len(sensitivity_train),
        "holdout_rows": len(sensitivity_holdout),
        "train_weather_complete_rows": int(
            sensitivity_train[weather_features].notna().all(axis=1).sum()
        ),
        "holdout_weather_complete_rows": int(
            sensitivity_holdout[weather_features].notna().all(axis=1).sum()
        ),
    },
])

display(feature_engineering_audit)

print("Feature contract:")
for key, value in feature_engineering_contract.items():
    print(f"{key}: {value}")

print("\nChecks:")
assert feature_engineering_contract["target"] == "Yield"
assert feature_engineering_contract["full_season_weather_alignment_verified"] is False
assert feature_engineering_contract["yield_area_production_units_verified"] is False
assert feature_engineering_contract["causal_claims_permitted"] is False
assert primary_train["FE_YearStart"].max() < primary_holdout["FE_YearStart"].min()

print("Contract checks passed.")

,cohort,train_rows,holdout_rows,train_weather_complete_rows,holdout_weather_complete_rows
0,primary,88551,18716,76317,16230
1,sensitivity,136622,30191,116775,25923


Feature contract:
notebook: 15_feature_engineering.ipynb
target: Yield
prediction_unit: district_crop_season_year
primary_cohort: POSITIVE_YIELD_MEASUREMENTS_POSITIVE
sensitivity_cohort: All positive finite non-aggregate yields
train_years: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
holdout_years: [2023, 2024]
split_strategy: Chronological holdout; no random-only split
core_categorical_features: ['Crop', 'Season', 'Crop_Category']
core_numeric_features: ['FE_YearStart']
exploratory_weather_features: ['tmin_mean_c', 'tmin_min_c', 'tmax_mean_c', 'tmax_max_c', 'precipitation_total_mm', 'precipitation_mean_mm_day', 'precipitation_max_mm_day', 'tmin_valid_days', 'tmax_valid_days', 'precipitation_valid_days']
weather_alignment: STARTING_CALENDAR_YEAR_PROXY
full_season_weather_alignment_verified: False
yield_area_production_units_verified: False
causal_claims_permitted: False

Checks:
Contract checks passed.


## Conclusion — Feature Engineering and Evaluation Readiness

This notebook established the target eligibility cohorts, reviewed potential inconsistencies in Area, Production, and Yield, audited weather coverage, and defined a chronological training/holdout split.

### Key outputs
- **Primary cohort:** 107,267 positive-yield records with positive Area and Production.
- **Sensitivity cohort:** 166,813 positive, finite, non-aggregate yield records.
- **Primary split:** 88,551 training records and 18,716 holdout records.
- **Sensitivity split:** 136,622 training records and 30,191 holdout records.
- **Core features:** Crop, Season, Crop_Category, and Year.
- **Exploratory weather features:** ten annual district-level weather variables.

### Limitations
- Yield measurement units have not been independently verified.
- Positive Area and Production do not guarantee that every yield observation is reliable.
- Weather coverage is geographically incomplete.
- Weather alignment uses a starting-calendar-year proxy; full-season alignment remains unverified.
- The chronological holdout evaluates later years, but does not independently test generalization to entirely unseen districts.
- No causal conclusions should be drawn from the planned predictive models.

### Handoff to Notebook 16
Begin with a simple historical baseline, then evaluate more complex models using the same chronological holdout. Compare core and weather-enhanced models on identical weather-complete evaluation rows, and report sensitivity-cohort results separately. Do not modify source measurements or silently remove extreme observations.